In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests, zipfile, io, os
import pandas as pd
import ast

In [ ]:
# Listings fetch
domain = "datasets.techmatrix.it/airml"
token = "DI_xeno_2026"

cities = ["sicilia", "trentino", "venezia", "roma", "puglia",
          "napoli", "firenze", "milano", "bergamo", "bologna"]

for city in cities:
    data_dir = os.path.join("./data", city)
    if os.path.exists(data_dir):
        print(f"Skipping {city}: folder already exists")
        continue
    
    url = f"https://{domain}/listings/{city}.zip?token={token}"
    resp = requests.get(url, stream=True)
    if resp.ok:
        os.makedirs(data_dir, exist_ok=True)
        zip_path = os.path.join("./data", f"{city}.zip")
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(path=data_dir)
        os.remove(zip_path)
    else:
        print(f"Failed to download {city}: {resp.status_code}")

In [ ]:
# Reviews fetch
for city in cities:
    data_dir = os.path.join("./data", city)
    if os.path.exists(data_dir):
        print(f"Skipping {city}: folder already exists")
        continue

    url = f"https://{domain}/reviews/{city}.zip?token={token}"
    resp = requests.get(url, stream=True)
    if resp.ok:
        if not os.path.exists(data_dir):
            os.makedirs(data_dir, exist_ok=True)
        zip_path = os.path.join("./data", f"{city}.zip")
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(8192):
                if chunk:
                    f.write(chunk)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(path=data_dir)
        os.remove(zip_path)
    else:
        print(f"Failed to download {city}: {resp.status_code}")

# Filtering

In [ ]:
COLS_TO_DROP = {
    # URL / immagini
    "listing_url", "picture_url", "host_thumbnail_url", "host_picture_url", "host_url",
    # Testuali / identificativi listing
    "name", "description", "neighborhood_overview", "calendar_updated",
    # Identificatori di scraping / metadati tecnici
    "scrape_id", "last_scraped", "source",
    # Identificatori personali / dati host
    "host_id", "host_name", "host_since", "host_location", "host_about",
    "host_neighbourhood", "host_listings_count", "host_total_listings_count",
    "host_verifications", "host_has_profile_pic", "host_identity_verified",
    # Metriche risposta host
    "host_response_time", "host_response_rate", "host_acceptance_rate", "host_is_superhost",
    # Location duplicate / non predittive
    "neighbourhood", "neighbourhood_group_cleansed",
    # Testo derivabile / calcolato
    "bathrooms_text", "first_review", "last_review",
    # Calcolati host (aggregati)
    "calculated_host_listings_count", "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms", "calculated_host_listings_count_shared_rooms",
}
dfs = []

for sub in os.listdir("./data"):
    subpath = os.path.join("./data", sub)
    if not os.path.isdir(subpath):
        continue

    csv_path = os.path.join(subpath, "listings.csv")

    if os.path.exists(csv_path):
        city_df = pd.read_csv(csv_path, usecols=lambda col: col not in COLS_TO_DROP)
        city_df["city"] = sub
        dfs.append(city_df)
    else:
        for root, _, files in os.walk(subpath):
            if "listings.csv" in files:
                city_df = pd.read_csv(os.path.join(root, "listings.csv"), usecols=lambda col: col not in COLS_TO_DROP)
                city_df["city"] = sub
                dfs.append(city_df)
                break

if dfs:
    listings = pd.concat(dfs, ignore_index=True)
else:
    listings = pd.DataFrame()

listings.info()

In [ ]:
REVIEW_COLS_TO_DROP = {
    "reviewer_name", "date"
}
review_dfs = []

for sub in os.listdir("./data"):
    subpath = os.path.join("./data", sub)
    if not os.path.isdir(subpath):
        continue
    csv_path = os.path.join(subpath, "reviews.csv")
    if os.path.exists(csv_path):
        review_dfs.append(pd.read_csv(csv_path, usecols=lambda col: col not in REVIEW_COLS_TO_DROP))
    else:
        for root, _, files in os.walk(subpath):
            if "reviews.csv" in files:
                review_dfs.append(pd.read_csv(os.path.join(root, "reviews.csv"), usecols=lambda col: col not in REVIEW_COLS_TO_DROP))
                break

if review_dfs:
    reviews = pd.concat(review_dfs, ignore_index=True)
else:
    reviews = pd.DataFrame()

reviews.info()

In [ ]:
# 1. Drop righe duplicate
dupes = listings.duplicated().sum()
listings = listings.drop_duplicates().reset_index(drop=True)
print(f"Righe duplicate rimosse: {dupes}")

In [ ]:

# 2. Drop righe con target nullo (price) o con più del 70% di valori nulli
null_price = listings["price"].isna().sum()
listings = listings.dropna(subset=["price"])
print(f"Righe con price nullo rimosse: {null_price}")

# Righe con più del 70% di valori nulli
thresh = int(0.70 * listings.shape[1])
sparse_mask = listings.isna().sum(axis=1) > thresh
sparse_count = sparse_mask.sum()
listings = listings[~sparse_mask].reset_index(drop=True)
print(f"Righe con >70% nulli rimosse: {sparse_count}")

In [ ]:
# 3. Drop righe con accommodates < 1
low_acc = (listings["accommodates"] < 1).sum()
listings = listings[listings["accommodates"] >= 1].reset_index(drop=True)
print(f"Righe con accommodates < 1 rimosse: {low_acc}")

### SAMPLE DEI DATI

Siccome i dati sono molti. Effettuiamo un campionamento casuale per velocizzare le operazioni di preprocessing e modellazione.

In [ ]:
listings_all = listings.copy()

In [ ]:
# Eseguire questa cella per avere tutti i dati
listings = listings_all.copy()

In [ ]:
listings = listings.sample(n=30000, random_state=7112004).reset_index(drop=True)

## 2. Analisi Esplorativa dei Dati

### 2.1 Statistiche Generali


Questa sottosezione calcola le statistiche descrittive per le colonne del dataframe `listings`.

Le celle sono suddivise in sotto-sezioni:

- **2.1.a** Esplorazione delle colonne.
- **2.1.b** Statistiche descrittive: media, std, quartili, conteggio valori mancanti e unici.
- **2.1.c** Top valori per alcune colonne categoriche.
- **2.1.d** Matrice di correlazione tra variabili numeriche e relativa heatmap.


#### 2.1.a Esplorazione delle colonne

Estraiamo le info relative alle colonne del dataframe `listings`.

In [ ]:
listings.info()

Il dataset contiene 43 variabili totali. Guardiamo il contenuto di queste variabili per capire meglio la distribuzione dei dati e identificare eventuali anomalie o valori mancanti.

In [ ]:
listings.head()

Estraiamo l'elenco delle colonne numeriche presenti nel dataset `listings` per identificare quali feature numeriche sono disponibili per l'analisi. Questo ci aiuterà a capire meglio la struttura dei dati e a pianificare le fasi successive dell'analisi esplorativa.

In [ ]:
listings.select_dtypes(include=[np.number]).info()

Il dataset presenta 33 colonne numeriche. Ecco la spiegazione delle principali colonne numeriche mostrate sopra:

- **accommodates**: numero massimo di ospiti supportati.
- **bathrooms**: numero di bagni.
- **bedrooms**: numero di camere da letto.
- **beds**: numero di letti disponibili.
- **minimum_nights / maximum_nights**: vincoli min/max di soggiorno.
- **availability_30 / availability_60 / availability_90 / availability_365**: giorni disponibili nei rispettivi intervalli.
- **number_of_reviews, reviews_per_month**: conteggio recensioni e frequenza mensile.
- **review_scores_rating**: punteggio medio delle recensioni (aggregato).
- **estimated_occupancy_l365d**: occupazione stimata su ultimi 365 giorni (target Task B).
- **estimated_revenue_l365d**: ricavo stimato annuo (se presente).
- **latitude / longitude**: coordinate geografiche (possono servire per mappe o distanza dal centro).
- **n_amenities**: (se calcolata) numero di servizi offerti dall'alloggio.

Le colonne non numeriche includono variabili categoriche (es. `neighbourhood_group`, `room_type`) e testuali (es. `name`, `description`), che richiederanno approcci di analisi e preprocessing differenti, che saranno affrontati nelle sezioni successive.

#### 2.1.b Statistiche descrittive numeriche

Applico il metodo describe a listing per avere una prima idea della distribuzione dei dati.

In [ ]:
listings.describe()

Questo metodo fornisce le seguenti statistiche per ogni colonna numerica:
- **count**: numero di osservazioni non-nulle.
- **mean / std**: media e deviazione standard; confrontale per capire dispersione e presenza di outlier.
- **min / 25% / 50% / 75% / max**: quantili utili per individuare asimmetrie e outlier (max >> 75% + IQR indica outlier).

Guardiamo ora quanti valori unici e quanti valori mancanti ci sono per ogni colonna numerica. Questo ci aiuterà a capire se ci sono variabili con molti valori mancanti.

In [ ]:
stats = pd.DataFrame()
stats['missing'] = listings.isna().sum()
stats['unique'] = listings.nunique()
display(stats)

Come si può vedere molte variabili hanno un numero elevato di valori mancanti, in particolare quelle relative alle recensioni. La maggior parte però ha un numero di valori mancanti molto basso, quindi possiamo considerare di convertire i valori mancanti o di rimuovere le righe con valori mancanti a seconda del caso specifico.

#### 2.1.c Categorie top values


In questa sezione esploriamo le colonne categoriche per identificare le categorie più frequenti. In particolare guardiamo le seguenti colonne:
- `neighbourhood_cleansed`: quartiere in cui si trova l'alloggio.
- `property_type`: tipo di proprietà (es. appartamento, casa, bed & breakfast).
- `room_type`: tipo di alloggio (es. intero appartamento, stanza privata).
- `amenities`: servizi offerti (es. Wi-Fi, cucina, aria condizionata).

Cominciamo estraendo delle statistiche relative a queste colonne categoriche per identificare le categorie più frequenti e capire meglio la distribuzione dei dati in queste variabili. Questo ci aiuterà a pianificare il preprocessing e a identificare categorie che potrebbero essere raggruppate o trattate in modo speciale.

In [ ]:
neighbourhoods = listings["neighbourhood_cleansed"].value_counts()
property_types = listings["property_type"].value_counts()
room_types = listings["room_type"].value_counts()
amenities = listings["amenities"].value_counts()

In [ ]:
print(neighbourhoods.head())
neighbourhoods.describe()

In [ ]:
print(property_types.head())
print(property_types.describe())

In [ ]:
print(room_types.head())
print(room_types.describe())

In [ ]:
print(amenities.head())
amenities.describe()

Come si può vedere, la variabile `neighbourhood_cleansed` ha 1011 valori unici, `property_type` ha 118 valori unici, `room_type` ha 4 valori unici e `amenities` ha 183716 valori unici.

`property_type` e `room_type` hanno un numero di categorie alto, quindi verranno presi i top 50 più frequenti, mentre per `room_type` che ha solo 4 categorie le visualizzeremo tutte.

per quanto riguarda `amenities`, è una variabile molto complessa, in quanto contiene una lista di servizi per record. Per questo motivo, è necessario analizzare questa variabile in modo diverso rispetto alle altre variabili categoriche. Estraiamo tutti i servizi presenti nel dataset e contiamo la frequenza di ogni servizio.

Proviamo ora a visualizzare la distribuzione di queste variabili categoriche tramite dei grafici a barre, per capire se ci sono delle categorie molto frequenti che potrebbero essere utili per il modello di regressione.

Adesso andremo a visualizzare la distribuzione di `amenities`. Per fare questo, siccome i valori in `amenities` sono una stringa che rappresenta una lista di stringhe, è necessario:
1. trasformare la stringa in una lista di stringhe
2. esplodere la lista in modo da avere una riga per ogni serivizio
3. contare la frequenza di ogni servizio

In questo modo è possibile ottenere un risultato simile a quello delle altre variabili categoriche, con la frequenza di ogni categoria presente nel dataset.

In [ ]:
all_amenities = listings["amenities"].apply(ast.literal_eval).explode().value_counts()
all_amenities

Anche in questo caso, avendo 15206 servizi unici, si visualizzeranno solo i 50 più frequenti, per avere un'idea della distribuzione dei servizi presenti nel dataset.

In [ ]:
top_amenities = all_amenities.head(50)
top_amenities.plot.bar(figsize=(14, 6), title="Top 50 Amenities")

Si può notare come le categorie hanno una distribuzione piuttosto uniforme, con nessuna categoria che rappresenta una percentuale eccessiva rispetto alle altre. Tuttavia è da considerare che questo campo contiene circa 16000 categorie uniche, quindi è possibile che alcune categorie siano rappresentate da un numero molto basso di istanze.

Nella parte del pre-processing, è possibile decidere di mantenere solo i servizi più frequenti, in modo da ridurre la dimensionalità del dataset e migliorare la performance del modello di regressione. Un altro dato utile potrebbe essere contare il numero di amenità presenti in ogni record, in modo da avere una variabile numerica che rappresenta la quantità di servizi offerti da ogni alloggio, che potrebbe essere ulteriormente utile per la predizione del prezzo.

In [ ]:
top_neighbourhoods = neighbourhoods.head(50)
top_neighbourhoods.plot.bar(figsize=(14, 6), title="Top 50 Neighbourhoods")

Per quanto riguarda i quartieri, le categorie più frequenti sono i centri storici delle città più grandi, in quanto sono le zone più turistiche e quindi più richieste per l'affitto di case vacanze. Oltre ai centri storici, sono presenti anche quartieri residenziali e quartieri periferici, che potrebbero essere meno richiesti ma comunque presenti nel dataset. I livelli di granularità dei quartieri sono diversi, con alcuni quartieri che rappresentano intere città, mentre altri rappresentano solo dei quartieri specifici all'interno di una città.

In [ ]:
top_property_types = property_types.head(50)
top_property_types.plot.bar(figsize=(14, 6), title="Top 50 Property Types")

Si può vedere come le categorie più frequenti di `property_type` siano concentrate su appartamenti, case vacanze e case indipendenti, che sono le tipologie di alloggio più richieste per l'affitto di case vacanze. Oltre a queste tipologie, sono presenti anche altre tipologie di alloggio meno richieste ma comunque presenti nel dataset.

In [ ]:
room_types.plot.bar(log=True)

Questo grafico mostra la distribuzione delle tipologie di alloggio presenti nel dataset. Come si può vedere, le tipologie più frequenti sono gli appartamenti, seguiti dalle stanze private. Il grafico è in scala logaritmica, in quanto ci sono solo un migliario di stanze d'hotel o stanze condivise, mentre per le altre tipologie di alloggio sono presenti decine di migliaia di istanze.

#### 2.1.d Correlation matrix and heatmap


In [ ]:
num_cols = listings.select_dtypes(include=[np.number]).columns.tolist()
corr = listings[num_cols].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation matrix (numeric features)')
plt.show()

Questa sottosezione visualizza la **matrice di correlazione** tra tutte le variabili numeriche del dataframe tramite una **heatmap**:

- Ogni cella della matrice misura la correlazione lineare tra due variabili numeriche.
- I valori vanno da **-1** a **1**:
  - **1** = correlazione positiva perfetta.
  - **0** = nessuna relazione lineare evidente.
  - **-1** = correlazione negativa perfetta.
- La diagonale principale è sempre pari a 1, perché ogni variabile è perfettamente correlata con sé stessa.

Come interpretare i colori della heatmap:

- Toni **rossi**: relazione positiva, cioè le due variabili tendono a crescere insieme.
- Toni **blu**: relazione negativa, cioè una cresce mentre l'altra tende a scendere.
- Toni molto chiari o quasi neutri: relazione debole o assente.

Cose interessanti da osservare nel nostro caso:

- Blocchi forti tra `accommodates`, `bathrooms`, `bedrooms` e `beds`: indicano che descrivono dimensioni simili dell'alloggio.
- Correlazioni tra `availability_30`, `availability_60`, `availability_90` e `availability_365`: sono attese perché misurano la disponibilità nello stesso senso ma su finestre diverse.
- Relazioni tra `number_of_reviews`, `reviews_per_month` e i punteggi recensione: utili per capire se l'attività dell'alloggio è associata alla qualità percepita.
- Eventuali correlazioni con `price` e `estimated_occupancy_l365d`: sono quelle più utili in vista dei modelli di regressione.

### 2.3 Specifiche per le singole task


Questa sezione contiene le specifiche per l'analisi esplorativa dei dati mirata alle singole task.

Le celle sono suddivise in sotto-sezioni:

- **2.3.a** Price Prediction
- **2.3.b** Occupancy Regression
- **2.3.c** NLP Classification
- **2.3.d** Recommendation System


#### 2.3.a - Price Regression

Per il task di regressione su `price` è necessario filtrare i dati per rimuovere le feature non necessarie. In particolare è necessario rimuovere le feature riguardanti il tasso di occupazione, in quanto non sono importanti per la predizione del prezzo.

Creiamo quindi un nuovo DataFrame `lst_for_price_analysis` che contiene solo le feature necessarie per la predizione del prezzo. In particolare, rimuoviamo le feature riguardanti il tasso di occupazione e la disponibilità:

- `minimum_nights`, `maximum_nights`, `minimum_minimum_nights`, `maximum_minimum_nights`, `minimum_maximum_nights`, `maximum_maximum_nights`, `minimum_nights_avg_ntm`, `maximum_nights_avg_ntm`
- `calendar_updated`, `has_availability`, `availability_30`, `availability_60`, `availability_90`, `availability_365`, `calendar_last_scraped`
- `number_of_reviews`, `number_of_reviews_ltm`, `number_of_reviews_l30d`, `availability_eoy`, `number_of_reviews_ly`
- `estimated_occupancy_l365d`, `estimated_revenue_l365d`


Manteniamo quindi solo le feature che potrebbero essere utili per la predizione del prezzo, ovvero:
- `id`
- `listing_url`
- `neighbourhood_cleansed`
- `latitude`
- `longitude`
- `property_type`
- `room_type`
- `accommodates`
- `bathrooms`
- `bedrooms`
- `beds`
- `amenities`
- `price`

In [ ]:
filtered_features = [
    "id",
    "neighbourhood_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "amenities",
    "price"
]

lst_for_price_analysis = listings[filtered_features]

lst_for_price_analysis.info()

Il nuovo DataFrame `lst_for_price_analysis` contiene quindi 12 variabili, di cui 11 sono feature e 1 è la variabile da predire. Le feature sono sono perlopiù numeriche. Ci sono alcune feature categoriche:
- `neighbourhood_cleansed`
- `property_type`
- `room_type`
- `amenities`

Che possono essere importanti per la predizione del prezzo, ma che vanno trattate in modo diverso dalle feature numeriche. In particolare, è necessario trasformare le feature categoriche viste nella sezione precedente in nuove variabili binarie.
Inoltre bisogna trattare anche il valore di `price`, che è una stringa. Per poter utilizzare questa variabile e poterla visualizzare nei plot, è necessario trasformarla in un valore numerico.

Vediamo ora il formato della variabile `price`.

In [ ]:
print(lst_for_price_analysis["price"])

Come si può vedere, la variabile `price` è una stringa che contiene il simbolo del dollaro e le virgole per le migliaia. Per poter utilizzare questa variabile e poterla visualizzare nei plot, è necessario trasformarla in un valore numerico:

In [ ]:
lst_for_price_analysis["price"] = (
    lst_for_price_analysis["price"]
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
)
lst_for_price_analysis["price"]

Adesso `price` è una variabile numerica che può essere utilizzata per la predizione del prezzo.

Ora applichiamo il metodo describe al nuovo DataFrame `lst_for_price_analysis` per avere una prima idea della distribuzione dei dati dopo il filtraggio.

In [ ]:
lst_for_price_analysis.describe()

La `latitudine`, come aspettato, è compresa tra 35.6 e 46.5, mentre la `longitudine` tra 9 e 18.5, che corrispondono alla posizione geografica dell'Italia. Per quanto riguarda le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds`, si nota che sono presenti dei valori outliers.

Difatti il numero massimo di `accomodates` è 16, ancora accettabile, ma ben più alto dal valore del percentile 75 che equivale a 5. Per di più il numero massimo di `bathrooms` è 100, di `bedrooms` è 44 e di `beds` è 50, che sono valori molto elevati e potrebbero essere considerati outliers risultato di errori di immissione.

Questi valori si discostano significativamente dai dati, e possono influenzare negativamente la performance del modello di regressione. Per questo motivo, è necessario trattarli prima di procedere con la fase di modellazione.

Per quanto riguarda la variabile `price`, si nota che il prezzo massimo è 80000, mentre il prezzo del 75-esimo percentile è 168. Questo indica che ci sono dei valori di prezzo molto elevati che potrebbero anche essi essere considerati outliers.

Prima di andare a visualizzare con dei plot la distribuzione delle variabili, trattiamo i valori mancanti, visti nella sezione precedente. Visualizziamo comunque per ogni variabile il numero di valori mancanti:

In [ ]:
na_values = lst_for_price_analysis.isna().sum()
print(na_values)

Come possiamo vedere le variabili `bathrooms`, `bedrooms` e `beds` contengono un numero poco significativo di valori mancanti rispetto ai dati totali. Si suppone che questi valori mancanti siano dovuti a errori di immissione, siccome è impossibile avere un alloggio con 0 camere o 0 letti. Per quanto riguarda i bagni, è possibile che ci siano alloggi senza bagno, ma è più probabile che si tratti di errori di immissione. 

Per questi motivi:
- Per le variabili `bedrooms` e `beds`, si suppone che i valori mancanti siano dovuti a errori di immissione e verranno tolti dal dataset.
- Per la variabile `bathrooms`, si suppone che effettivamente ci siano alloggi senza bagno, quindi i valori mancanti verranno sostituiti con 0.

In [ ]:
lst_for_price_analysis = lst_for_price_analysis.dropna(subset=["bedrooms", "beds"])
lst_for_price_analysis["bathrooms"] = lst_for_price_analysis["bathrooms"].fillna(0)
na_values = lst_for_price_analysis.isna().sum()
print(na_values)

Ora non abbiamo più valori mancanti nelle variabili `bathrooms`, `bedrooms` e `beds`, e possiamo procedere con la visualizzazione della distribuzione delle variabili tramite dei plot.

Come primo plot, è possibile visualizzare la distribuzione della variabile `price` tramite un istogramma. In questo modo è possibile vedere se ci sono dei valori di prezzo molto elevati che potrebbero essere considerati outliers.

In [ ]:
lst_for_price_analysis["price"].plot.hist(
    bins=50,
    log=True,
)

Come si può vedere, la distribuzione della variabile `price` è molto sbilanciata, con la maggior parte dei valori concentrati tra 0 e 10000, con alcuni valori molto elevati che potrebbero essere considerati outliers. Decidiamo di togliere i valori estremi, in particolare sopra i 10000 e sotto i 5. Il grafico è in scala logaritmica, in modo tale che anche i valori più elevati siano visibili. Questa cosa può essere vista anche con il boxplot:

In [ ]:
lst_for_price_analysis["price"] = lst_for_price_analysis["price"].clip(upper=10000, lower=5)
lst_for_price_analysis["price"].plot.hist(
    bins=50,
    log=True,
)

Ora si può vedere che la distribuzione di price si concentra su due picchi principali, uno intorno a 5-2000 e l'altro intorno a 7000-1000. Questo potrebbe indicare che ci sono due categorie principali di alloggi, alcuni più economici e altri che potrebbero essere considerati di lusso.

Controlliamo ora il boxplot della variabile `price` per identificare visivamente la presenza di outliers e la distribuzione dei prezzi.

In [ ]:
lst_for_price_analysis["price"].plot.box()

Proviamo a creare un istogramma colorato in base al `room_type`, per vedere se ci sono differenze nella distribuzione dei prezzi in base al tipo di alloggio.

In [ ]:
# Proviamo a creare un istogramma colorato in base al `room_type`, per vedere se ci sono differenze nella distribuzione dei prezzi in base al tipo di alloggio.
room_types = lst_for_price_analysis["room_type"].values
plt.figure(figsize=(10, 6))
sns.histplot(data=lst_for_price_analysis, x="price", hue=room_types, bins=50, log_scale=True)
plt.title("Distribuzione dei prezzi per room_type")
plt.xlabel("Price (log scale)")
plt.ylabel("Count")
plt.legend(title="Room Type")
plt.show()

Generiamo ora degli istogrammi per le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds` per visualizzare meglio la presenza di outliers.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds"]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel() # Appiattisce la matrice di assi in un array 1D per iterare più facilmente

for ax, col in zip(axes, cols):
    lst_for_price_analysis[col].dropna().plot.hist(
        bins=50,
        log=True,
        ax=ax,
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Come si può vedere bathrooms, bedrooms e beds presentano dei valori molto elevati che sono il risultato di probabili errori di immissione. Per questo motivo questi valori saranno eliminati dal dataset:
- `bathrooms` < 20
- `bedrooms` < 22
- `beds` < 35

In [ ]:
lst_wout_outl_p_a = lst_for_price_analysis[
    (lst_for_price_analysis["bathrooms"] < 20) &
    (lst_for_price_analysis["bedrooms"] < 22) &
    (lst_for_price_analysis["beds"] < 35)
]

Adesso generiamo i boxplot per tutte le variabili per visualizzare il risulato del filtraggio degli outliers dati da palesi errori di immissione.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds", "price"]

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel() # Appiattisce la matrice di assi in un array 1D per iterare più facilmente

for ax, col in zip(axes, cols):
    lst_wout_outl_p_a[col].dropna().plot.box(
        ax=ax,
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

plt.tight_layout()
plt.show()

Ora proviamo a visualizzare la relazione tra `price` e le variabili `accomodates`, `bathrooms`, `bedrooms` e `beds` tramite dei scatter plot. In questo modo è possibile vedere se ci sono delle relazioni tra queste variabili e il prezzo, e se ci sono dei valori di prezzo molto elevati che potrebbero essere considerati outliers.

In [ ]:
cols = ["accommodates", "bathrooms", "bedrooms", "beds"]
colors = ["blue", "orange", "green", "red"]

sample = lst_wout_outl_p_a.sample(n=5000, random_state=7112004)

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel()

for ax, col in zip(axes, cols):
    sample[["price", col]].plot.scatter(
        x=col,
        y="price",
        ax=ax,
        logy= True,
        c=colors[cols.index(col)]
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Price")

plt.tight_layout()
plt.show()

In tutti e quattro i grafici si vede una tendenza positiva, anche se debole: all'aumentare del numero di `accomodates`, `bathrooms`, `bedrooms` e `beds`, aumenta anche il prezzo. Sicuramente variabili categoriche che abbiamo lasciato fuori da questa analisi, come `neighbourhood_cleansed`, `property_type`, `room_type` e `amenities`, possono avere anche esse un impatto sul prezzo, e potrebbero essere utili per migliorare la performance del modello di regressione.

Adesso analizziamo queste variabili. Estraiamo i valori unici per ogni variabile categorica:

Le ultime variabili da esplorare che possono essere utili per la predizione del prezzo sono le densità di letti, camere e bagni per ogni alloggio. Aggiungiamo quindi quattro nuove variabili al dataset:
- `beds_per_person` = `beds` / `accomodates`: il numero di letti per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più confortevole, e quindi potrebbe essere associato a un prezzo più elevato.
- `bedrooms_per_person` = `bedrooms` / `accomodates`: il numero di camere da letto per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più spazioso, e quindi potrebbe essere associato a un prezzo più elevato.
- `bathrooms_per_person` = `bathrooms` / `accomodates`: il numero di bagni per persona. Un valore più alto di questa variabile potrebbe indicare un alloggio più confortevole, e quindi potrebbe essere associato a un prezzo più elevato.
- `beds_per_bedroom` = `beds` / `bedrooms`: il numero di letti per camera da letto. Un valore più alto di questa variabile potrebbe indicare un alloggio più spazioso, e quindi potrebbe essere associato a un prezzo più elevato.

In [ ]:
data = lst_wout_outl_p_a.copy()
data["beds_per_person"] = data["beds"] / data["accommodates"]
data["bedrooms_per_person"] = data["bedrooms"] / data["accommodates"]
data["bathrooms_per_person"] = data["bathrooms"] / data["accommodates"]
data["beds_per_bedroom"] = data["beds"] / data["bedrooms"]
print(data)

Ora creiamo un grafico nello stile di quello superiore, per visualizzare se ci sono delle relazioni tra queste nuove variabili e il prezzo.

In [ ]:
cols = ["beds_per_person", "bedrooms_per_person", "bathrooms_per_person", "beds_per_bedroom"]
sample = data.sample(n=7000, random_state=7112004)

fig, axes = plt.subplots(len(cols) // 2 + len(cols) % 2, len(cols) // 2, figsize=(12, 10))
axes = axes.ravel()

for ax, col in zip(axes, cols):
    sample[["price", col]].plot.scatter(
        x=col,
        y="price",
        ax=ax,
        logy= True,
        logx= True,
        c=colors[cols.index(col)]
    )
    ax.set_title(col.capitalize())
    ax.set_xlabel(col)
    ax.set_ylabel("Price")

plt.tight_layout()
plt.show()

Tutte e quattro le variabili non mostrano una relazione molto forte con il prezzo. Per una visualizzazione più chiara è stato utilizzato il logaritmo anche dell'asse x. Possono però essere effettuate le seguenti osservazioni:
- `beds_per_person` e `bedrooms_per_person`: si nota una lieve concentrazione dei prezzi più elevati in corrispondenza di valori prossimi a 1, ovvero listing con un letto o una camera per ospite. Tuttavia la dispersione è elevata, quindi il potere predittivo di queste feature è limitato.
- `bathrooms_per_person`: non emerge alcuna tendenza chiara. La distribuzione è sostanzialmente piatta lungo l'asse x, indicando una scarsa correlazione con il prezzo.
- `beds_per_bedroom`: si osserva una leggera tendenza inversa, infatti listing con più letti per camera (ipoteticamente dormitori o strutture condivise) tendono ad avere prezzi più bassi. Questa feature potrebbe quindi catturare indirettamente anche la tipologia di alloggio.

#### 2.3.b - Occupancy Regression

Obiettivo: esplorare `estimated_occupancy_l365d` e le sue relazioni con le feature disponibili, escludendo le colonne che creerebbero leakage: `price`, `estimated_revenue_l365d`, tutte le colonne `availability_*` e le colonne `number_of_reviews*`.

Sottosezioni:

- **A**: Panoramica target (distribuzione, missingness, statistica descrittiva).
- **B**: Correlazioni numeriche — elenco feature più correlate (positive/negative) con il target.
- **C**: Scatter / regplot per le top feature correlate.
- **D**: Analisi categorica: boxplot di `estimated_occupancy_l365d` per `room_type` e per i top quartieri.
- **E**: Missingness per feature rilevanti.

In [ ]:
# A Target overview: `estimated_occupancy_l365d`
t = listings['estimated_occupancy_l365d']
print('Count non-null:', t.count())
print('Missing:', t.isna().sum())
display(t.describe())
plt.figure(figsize=(8,4))
sns.histplot(t.dropna(), kde=True, bins=50)
plt.title('Distribution of estimated_occupancy_l365d')
plt.xlabel('estimated_occupancy_l365d')
plt.show()
plt.figure(figsize=(6,3))
sns.boxplot(x=t)
plt.title('Boxplot of estimated_occupancy_l365d')
plt.show()

### Come leggere i grafici

- Istogramma + KDE: mostra la forma della distribuzione (asimmetria, picchi, code).
- Boxplot: mette in evidenza mediana, IQR e outlier.
- Statistiche rapide: media, mediana, quartili, count e missing.

### Cosa emerge

- Distribuzione fortemente asimmetrica con massa vicino a 0 e lunga coda verso valori alti: molti annunci hanno poche prenotazioni mentre pochi sono molto popolari (hotel, appartamenti centrali).
- Mediana relativamente bassa: la maggioranza degli annunci presenta occupazione contenuta.
- Presenza di outlier (valori molto alti) che vanno verificati ma non sono necessariamente errori.
- Potrebbe essere utile considerare trasformazioni del target o modelli robusti se si cerca stabilità nelle previsioni.


In [ ]:
# B Correlazioni con il target
exclude_prefixes = ['availability_', 'number_of_reviews']
exclude_exact = {'price', 'estimated_revenue_l365d'}
num_cols = listings.select_dtypes(include=[np.number]).columns.tolist()
if 'estimated_occupancy_l365d' in num_cols:
    num_cols.remove('estimated_occupancy_l365d')
cols_for_corr = [c for c in num_cols if c not in exclude_exact and not any(c.startswith(p) for p in exclude_prefixes)]
print(f'Total numeric features considered for correlation: {len(cols_for_corr)}')
corr_with_target = listings[cols_for_corr + ['estimated_occupancy_l365d']].corr()['estimated_occupancy_l365d'].drop('estimated_occupancy_l365d')
corr_sorted = corr_with_target.sort_values(ascending=False)
display(corr_sorted.head(20))
display(corr_sorted.tail(20))

### Come leggere

- La tabella ordina le feature per correlazione con `estimated_occupancy_l365d` (dal più positivo al più negativo).
- Guardare sia il segno (positivo/negativo) sia la magnitudo: le feature con valore assoluto più alto sono le più rilevanti.
- Sono state escluse le colonne di possibile leakage: `price`, `estimated_revenue_l365d`, `availability_*` e `number_of_reviews*`.

### Cosa emerge

- Le metriche di recensione (es. `reviews_per_month`, `review_scores_*`) mostrano le associazioni più evidenti.
- Gli effetti osservati sono generalmente moderati: le correlazioni non sono enormi e possono essere influenzate da confondenti o da differenze fra città.
- Usare queste indicazioni per selezionare le candidate feature da approfondire con scatterplot e analisi stratificate.


In [ ]:
# C Scatter / regplot per le top feature correlate
corr_vals = corr_with_target.abs().sort_values(ascending=False)
top_feats = corr_vals.head(5).index.tolist()
print('Top features by absolute correlation:', top_feats)

n_cols = 3
n_rows = 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 10))
axes = axes.ravel()

for ax, f in zip(axes, top_feats):
    sns.regplot(x=listings[f], y=listings['estimated_occupancy_l365d'], scatter_kws={'alpha':0.35}, ax=ax)
    ax.set_title(f'{f} vs estimated_occupancy_l365d')
    ax.set_xlabel(f)
    ax.set_ylabel('estimated_occupancy_l365d')

for ax in axes[len(top_feats):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

### Come leggere gli scatter/regplot

- Ogni grafico mostra la relazione tra una feature numerica e `estimated_occupancy_l365d`.
- I punti sparsi danno l'idea della variabilità reale dei dati, mentre la linea di regressione aiuta a capire se il legame è crescente, decrescente o quasi assente.
- Se la nuvola di punti è molto larga e la linea è quasi orizzontale, la feature è poco informativa da sola.
- La vista a scacchiera rende più facile confrontare i grafici e vedere subito quali relazioni sembrano più stabili.

### Cosa emerge

- Le feature più interessanti sono `reviews_per_month`, `latitude`, `longitude`, `review_scores_value` e `review_scores_accuracy`, ma il segnale non è sempre pulito.
- `reviews_per_month` sembra la più informativa, anche se la relazione è influenzata da alcuni valori molto estremi e da una forte concentrazione di punti nei valori bassi.
- Le variabili geografiche vanno lette con cautela: il dataset è aggregato per città, quindi `latitude` e `longitude` riflettono anche differenze tra mercati diversi.
- In generale, questi grafici servono soprattutto a individuare feature promettenti e a capire la forma della relazione, non a dire che una variabile è forte in senso assoluto.

In [ ]:
# D Analisi categorica (semplificata): room_type + neighbourhoods filtrati
min_count = 10

# Boxplot per `room_type`
plt.figure(figsize=(10,4))
order = listings['room_type'].value_counts().index
sns.boxplot(x='room_type', y='estimated_occupancy_l365d', data=listings, order=order)
plt.xticks(rotation=45)
plt.title('Occupancy by room_type (top categories)')
plt.show()

# Boxplot per `neighbourhood_cleansed` ma escludendo quartieri con pochi listing
counts = listings['neighbourhood_cleansed'].value_counts()
good_neigh = counts[counts >= min_count].index
print(f'Plotting neighbourhoods with >= {min_count} listings ({len(good_neigh)} neighbourhoods).')
top_to_plot = counts.loc[good_neigh].sort_values(ascending=False).head(15).index
if len(top_to_plot) > 0:
    plt.figure(figsize=(12,5))
    sns.boxplot(x='neighbourhood_cleansed', y='estimated_occupancy_l365d',
                data=listings[listings['neighbourhood_cleansed'].isin(top_to_plot)],
                order=top_to_plot)
    plt.xticks(rotation=45)
    plt.title(f'Occupancy by neighbourhood (min_count={min_count}) - top {len(top_to_plot)}')
    plt.show()

counts = listings['neighbourhood_cleansed'].value_counts()
filtered = counts[counts >= min_count].sort_values(ascending=False)
print(filtered.to_string())

### Come leggere i boxplot

- Ogni boxplot confronta la distribuzione di `estimated_occupancy_l365d` tra le categorie (es. `room_type` o `neighbourhood_cleansed`).
- La linea centrale è la mediana; il box mostra l'IQR (50% centrale). Baffi e punti fuori indicano variabilità e outlier.
- Attenzione alle categorie con pochi annunci: medie elevate possono dipendere da pochi valori estremi.
- In questa versione abbiamo applicato un filtro di supporto: vengono plottati solo i quartieri con almeno `min_count` listing (default 10). Controllare sempre il valore di `min_count` usato nella cella di codice.
- Controllare sempre la frequenza (count) per categoria insieme alla mediana prima di trarre conclusioni.

### Cosa emerge

- Alcuni quartieri presentano medie molto alte (es. Lenna, Pra' Secco, Ca' Brentelle): prima di considerare questi risultati, verificare il conteggio di listing per quei quartieri — se il conteggio è basso, la mediana o la media può essere fuorviante.
- `room_type` mostra differenze pratiche: intere case tendono ad avere mediane e variabilità maggiori rispetto a stanze condivise o private.
- Il filtro `min_count` riduce l'impatto dei quartieri con supporto scarso, rendendo le comparazioni più robuste; tuttavia potrebbe escludere nicchie reali se `min_count` è troppo alto.
- Questi grafici servono a individuare categorie e quartieri da approfondire con analisi stratificate o con filtri per numero di osservazioni; usare la stampa dei conteggi (ordinata per frequenza) per ispezionare i quartieri esclusi.


In [ ]:
# E Missingness summary per feature rilevanti
relevant = cols_for_corr.copy() if 'cols_for_corr' in globals() else []
relevant += ['estimated_occupancy_l365d'] if 'estimated_occupancy_l365d' in listings.columns else []
miss_pct = listings[relevant].isna().mean().sort_values(ascending=False)
print('Missingness percentage for relevant numeric features:')
display(miss_pct.head(20))
plt.figure(figsize=(6,8))
sns.heatmap(listings[relevant].isna().astype(int).sample(frac=0.2, random_state=42).T, cbar=False)
plt.title('Missingness heatmap (sampled rows)')
plt.show()

### Come leggere la missingness

- La tabella mostra la percentuale di valori mancanti per ciascuna feature: valori più alti indicano meno osservazioni utili per quella colonna.
- La heatmap campionata visualizza la struttura dei missing (colonne = variabili, righe = campione di record): strisce verticali indicano missing sparsi, blocchi orizzontali indicano assenze sistematiche su gruppi di righe.
- Confrontare sempre missingness e target: se il target è completo, le feature mancanti possono essere imputate o segnalate con indicatori di missing.
- Prestare attenzione a variabili derivate da recensioni (`review_scores_*`, `reviews_per_month`): spesso hanno missing non casuale (dipendono dall'assenza di recensioni).

### Cosa emerge

- Le feature legate alle recensioni mostrano missing non trascurabile: considerare imputazione (mediana) o l'uso di una variabile indicator per missingness.
- La heatmap suggerisce pattern sparsi piuttosto che blocchi grandi: i missing tendono a essere associati a singoli annunci, non a interi sottogruppi (ma verificare stratificando per città/quartiere).
- Regole pratiche consigliate: valutare l'eliminazione di colonne con percentuale di missing molto alta (es. >50%), oppure imputare con strategie stratificate; testare modelli con/senza le colonne imputate.
- Se i missing sono correlati a feature geografiche o categorie, preferire imputazione stratificata per evitare bias.


In [ ]:
# F Caricamento dei dati di calendar precedentemente processati
calendar_dfs = []

for sub in os.listdir("./data"):
    subpath = os.path.join("./data", sub)
    if not os.path.isdir(subpath):
        continue

    csv_path = os.path.join(subpath, "calendar.csv")

    if os.path.exists(csv_path):
        city_df = pd.read_csv(csv_path)
        calendar_dfs.append(city_df)
    else:
        for root, _, files in os.walk(subpath):
            if "calendar.csv" in files:
                city_df = pd.read_csv(os.path.join(root, "calendar.csv"))
                calendar_dfs.append(city_df)
                break

if calendar_dfs:
    calendars = pd.concat(calendar_dfs, ignore_index=True)
else:
    calendars = pd.DataFrame()

calendars.count()

Sezione di esplorazione dei dati di `calendar`: controlliamo dimensioni, intervallo temporale, distribuzione delle variabili di occupazione e qualche statistica base sui campi più utili per il Task B.  
L'obiettivo è avere un primo quadro della disponibilità giornaliera e del tasso di occupazione osservato, senza entrare ancora nel preprocessing vero e proprio.

In [ ]:
print('Shape:', calendars.shape)
display(calendars.head(1))

if 'date' in calendars.columns:
    calendars['date'] = pd.to_datetime(calendars['date'], errors='coerce')
    print('Date range:', calendars['date'].min(), '->', calendars['date'].max())

numeric_cols = [c for c in ['occ_days_01', 'occ_days_02', 'occ_days_03', 'occ_days_04', 'occ_days_05', 'occ_days_06', 'occ_days_07', 'occ_days_08', 'occ_days_09', 'occ_days_10', 'occ_days_11', 'occ_days_12', 'occupancy_rate_01', 'occupancy_rate_02', 'occupancy_rate_03', 'occupancy_rate_04', 'occupancy_rate_05', 'occupancy_rate_06', 'occupancy_rate_07', 'occupancy_rate_08', 'occupancy_rate_09', 'occupancy_rate_10', 'occupancy_rate_11', 'occupancy_rate_12', 'total_occ_days', 'total_observed_days', 'monthly_occupancy_mean', 'occupancy_rate'] if c in calendars.columns]

if numeric_cols:
    display(calendars[numeric_cols].describe().T)

**Correlazione tra le metriche di occupazione:**

In [ ]:
summary_cols = [c for c in ['monthly_occupancy_mean', 'occupancy_rate', 'total_occ_days', 'total_observed_days'] if c in calendars.columns]
if summary_cols:
    corr_calendar = calendars[summary_cols].corr()
    plt.figure(figsize=(6, 4))
    sns.heatmap(corr_calendar, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
    plt.title('Correlation matrix for calendar summaries')
    plt.tight_layout()
    plt.show()

**Confronto Occupancy rate nei vari mesi**

In [ ]:
rate_cols = [c for c in calendars.columns if c.startswith('occupancy_rate_')]
if rate_cols:
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    axes = axes.ravel()
    for ax, col in zip(axes, rate_cols):
        sns.histplot(calendars[col].dropna(), bins=40, kde=True, ax=ax)
        ax.set_title(col)
    plt.tight_layout()
    plt.show()

Dall'esplorazione minima emergono alcuni punti utili per il Task B:
- `calendar` contiene 193940 righe e 29 colonne, quindi il dataset è già abbastanza ricco da solo.
- Le distribuzioni di `occupancy_rate_01`, `occupancy_rate_02` e `occupancy_rate_03` sono fortemente bimodali, con molta massa vicino a 0 e a 1: questo suggerisce che molti annunci sono quasi sempre liberi o quasi sempre occupati nei mesi osservati.
- I valori osservati nei giorni mensili variano poco attorno al totale atteso, quindi il dato calendar sembra stabile a livello di copertura temporale.
- In ottica modellistica, `monthly_occupancy_mean`, `occupancy_rate` e `total_occ_days` sembrano le variabili calendar più promettenti da confrontare con il target `estimated_occupancy_l365d`.

### 2.4 Task D — Review Intelligence Pipeline (Feature Extraction)

This section implements the **Review Intelligence Pipeline** to extract predictive, structured features from the guest reviews text (`reviews.csv`) for all target cities.

The extracted signals include:
- **VADER Sentiment**: `sentiment_mean`, `sentiment_std`, `sentiment_min`, `pct_positive`, `pct_negative`.
- **Domain Lexicons (Aspects)**: cleanliness, location, value, comfort, host quality, accuracy.
- **Temporal signals**: active span, average intervals, sentiment trend, recency, acceleration.
- **Metadata**: average review length, standard deviation, word counts.
- **NMF Topic Modeling**: topic distributions (`topic_0` to `topic_4`) and `dominant_topic`.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Lexicon mappings for Aspect Extraction
ASPECTS = {
    "cleanliness":  ["clean", "dirty", "spotless", "tidy", "dust", "stain", "hygienic", "smell", "mold", "filthy", "immaculate"],
    "location":     ["location", "central", "metro", "station", "walk", "close", "far", "noisy", "quiet", "neighborhood", "convenient", "transport", "bus", "tram"],
    "value":        ["value", "price", "expensive", "cheap", "worth", "overpriced", "bargain", "money", "affordable"],
    "comfort":      ["comfortable", "cozy", "spacious", "cramped", "bed", "mattress", "pillow", "sleep", "noise", "thin walls"],
    "host_quality": ["host", "responsive", "helpful", "kind", "welcoming", "rude", "communication", "attentive", "friendly", "flexible", "accommodating"],
    "accuracy":     ["accurate", "photos", "description", "misleading", "exactly", "as described", "different", "expectation"]
}

# Multilingual Stopwords to force semantic (rather than language-based) topics
ITALIAN_STOPS = {
    "il", "lo", "la", "le", "gli", "un", "una", "di", "da", "in",
    "con", "su", "per", "tra", "fra", "che", "non", "è", "sono",
    "molto", "tutto", "questa", "questo", "anche", "più", "ma"
}
GERMAN_STOPS = {
    "der", "die", "das", "und", "ist", "in", "zu", "den", "von", "mit", 
    "auf", "für", "was", "waren", "war", "wir", "es", "ein", "eine", "sich", 
    "dem", "dass", "er", "sie", "uns", "aus", "an", "bei", "noch", "nur", "oder", "sehr"
}
FRENCH_STOPS = {
    "le", "la", "les", "et", "en", "de", "dans", "une", "un", "est", 
    "pour", "nous", "vous", "avec", "tout", "plus", "sur", "très", "était", "qui", "aux"
}
SPANISH_STOPS = {
    "el", "la", "los", "las", "y", "en", "de", "un", "una", "es", 
    "para", "con", "por", "lo", "nos", "se", "al", "del", "que", "muy", "todo", "como"
}
COMBINED_STOPS = list(ENGLISH_STOP_WORDS | ITALIAN_STOPS | GERMAN_STOPS | FRENCH_STOPS | SPANISH_STOPS)
COMBINED_STOPS.append("br")

# Text cleaning function: removes HTML residues and isolated 'br' words
def clean_review_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'<br\s*/?>', ' ', text)  # Replace HTML breaks
    text = re.sub(r'[^a-záàéèíìóòúùñü\s]', ' ', text)
    text = re.sub(r'\bbr\b', ' ', text)      # Remove isolated 'br'
    return re.sub(r'\s+', ' ', text).strip()

print("Task D Setup complete. Combined stops count:", len(COMBINED_STOPS))


In [ ]:
# Loop and extract structured features per city to preserve memory and handle the ~2.5 GB dataset
city_features = []
listing_texts = {}
analyzer = SentimentIntensityAnalyzer()
MAX_REVIEWS_PER_LISTING = 200  # For scaling performance, sample up to 200 recent reviews per listing

for sub in os.listdir("./data"):
    subpath = os.path.join("./data", sub)
    if not os.path.isdir(subpath):
        continue

    csv_path = os.path.join(subpath, "reviews.csv")
    if not os.path.exists(csv_path):
        # Check subdirectories as well
        for root, _, files in os.walk(subpath):
            if "reviews.csv" in files:
                csv_path = os.path.join(root, "reviews.csv")
                break

    if os.path.exists(csv_path):
        print(f"Processing reviews for city: {sub}...")
        
        # Load reviews dynamically with only required columns
        df = pd.read_csv(csv_path, usecols=["listing_id", "comments", "date"])
        df = df.dropna(subset=["comments"])
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"])
        
        # Sort reviews chronologically per listing and cap them to recent ones to optimize performance
        df = df.sort_values(["listing_id", "date"], ascending=[True, False])
        df = df.groupby("listing_id").head(MAX_REVIEWS_PER_LISTING).copy()
        
        # Apply text cleaning
        df["comments_clean"] = df["comments"].apply(clean_review_text)
        
        # VADER Sentiment Analysis
        scores = [analyzer.polarity_scores(c) for c in df["comments_clean"].tolist()]
        df["vader_compound"] = [s["compound"] for s in scores]
        df["vader_neg"] = [s["neg"] for s in scores]
        
        # Aspect keyword counts (optimized using vectorized C-level regex engines)
        for aspect, keywords in ASPECTS.items():
            pattern = r'\b(' + '|'.join(keywords) + r')\b'
            df[f"asp_{aspect}"] = df["comments_clean"].str.count(pattern).fillna(0)
            
        # Text Metadata
        df["review_length"] = df["comments"].str.len().fillna(0)
        df["review_word_count"] = df["comments"].str.split().str.len().fillna(0)
        
        # Store concatenated text per listing for NMF Topic Modeling
        for lid, group in df.groupby("listing_id"):
            all_text = " ".join(group["comments_clean"].tolist())
            listing_texts[lid] = listing_texts.get(lid, "") + " " + all_text
            
        # Fast Groupby Aggregations
        aggregations = {
            "vader_compound": ["mean", "std", "min"],
            "review_length": ["mean", "std", "max"],
            "review_word_count": ["mean"]
        }
        for aspect in ASPECTS:
            aggregations[f"asp_{aspect}"] = ["mean"]
            
        agg_df = df.groupby("listing_id").agg(aggregations)
        # Flatten multi-level columns
        agg_df.columns = [f"{col[0]}_{col[1]}" for col in agg_df.columns]
        
        # Calculate ratio of positive/negative reviews
        pct_df = df.groupby("listing_id")["vader_compound"].agg(
            pct_positive=lambda x: (x > 0.05).mean(),
            pct_negative=lambda x: (x < -0.05).mean()
        )
        
        # Vectorized Temporal Signals
        temp_agg = df.groupby("listing_id")["date"].agg(["min", "max", "count"])
        temp_agg["review_span_days"] = (temp_agg["max"] - temp_agg["min"]).dt.days
        temp_agg["avg_days_between_reviews"] = temp_agg["review_span_days"] / (temp_agg["count"] - 1 + 1e-9)
        
        # Days since last review (relative to scrape date 2025-09-22)
        temp_agg["days_since_last_review"] = (pd.Timestamp("2025-09-22") - temp_agg["max"]).dt.days
        
        # Join all aggregated features for the city
        city_agg = agg_df.join(pct_df).join(temp_agg[["review_span_days", "avg_days_between_reviews", "days_since_last_review"]])
        city_features.append(city_agg)

# Concatenate listing-level features across all processed cities
if city_features:
    consolidated_df = pd.concat(city_features, axis=0)
    # Handle duplicated listings across different city downloads if any
    consolidated_df = consolidated_df.groupby(level=0).mean()
    print("Aggregated reviews shape:", consolidated_df.shape)
else:
    consolidated_df = pd.DataFrame()
    print("No reviews processed.")

In [ ]:
# Phase 3: TF-IDF and NMF Topic Modeling across all listings
if not consolidated_df.empty and listing_texts:
    print("Fitting TF-IDF and NMF Topic Modeling...")
    
    # Filter listing_texts to only include listings present in consolidated_df
    active_lids = consolidated_df.index.tolist()
    active_texts = [listing_texts.get(lid, "") for lid in active_lids]
    
    # Fit TF-IDF Vectorizer (increased min_df to 20 to decrease sparsity)
    tfidf = TfidfVectorizer(max_features=5000, stop_words=COMBINED_STOPS, ngram_range=(1, 2), min_df=20)
    tfidf_matrix = tfidf.fit_transform(active_texts)
    
    # Fit NMF Topic Modeling using mathematically stable nndsvda initialization to avoid overflow/underflow warnings
    N_TOPICS = 5
    nmf = NMF(n_components=N_TOPICS, random_state=42, max_iter=300, init="nndsvda")
    topic_matrix = nmf.fit_transform(tfidf_matrix)
    
    # Topic Dataframe
    topic_df = pd.DataFrame(
        topic_matrix,
        columns=[f"topic_{i}" for i in range(N_TOPICS)],
        index=active_lids
    )
    topic_df["dominant_topic"] = topic_df[[f"topic_{i}" for i in range(N_TOPICS)]].idxmax(axis=1)
    # Convert to numerical topic category index
    topic_df["dominant_topic"] = topic_df["dominant_topic"].str.replace("topic_", "").astype(int)
    
    # Print top words per topic to verify coherence
    feature_names = tfidf.get_feature_names_out()
    for i, component in enumerate(nmf.components_):
        top_words = [feature_names[j] for j in component.argsort()[-15:]]
        print(f"Topic {i} Top Words: {', '.join(top_words)}")
        
    # Consolidate all NLP features
    nlp_features_df = consolidated_df.join(topic_df)
    print("NLP Features shape:", nlp_features_df.shape)
    
    # Merge NLP features back into the global listings DataFrame
    original_count = len(listings)
    listings = listings.merge(nlp_features_df, left_on="id", right_index=True, how="left")
    
    # Impute missing review values with 0 for listings that have no reviews
    numeric_nlp_cols = nlp_features_df.select_dtypes(include=[np.number]).columns.tolist()
    listings[numeric_nlp_cols] = listings[numeric_nlp_cols].fillna(0)
    
    print(f"Successfully merged NLP features into global listings. Count check: {original_count} -> {len(listings)}")
else:
    print("Skipping topic modeling (no reviews loaded).")


### 2.5 Baseline Downstream XGBoost & Ablation Study

Now that the Review Intelligence Pipeline features are merged directly into the global `listings` DataFrame, they will be inherited by the Price and Occupancy data preprocessors. 
We will add baseline XGBoost estimators and conduct an ablation study to test the predictive lift of these NLP features.

In [ ]:
# Ablation Study for Task A - Price Prediction
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score

# Run the Preprocessing Pipeline for Price Prediction
print("Re-running preprocess_data with newly added review features...")
X_train_nlp, X_val_nlp, y_train_nlp, y_val_nlp, cols_nlp, preprocessor_nlp = preprocess_data(
    lst_for_price_preproc, 
    cols_to_drop=["neighbourhood", "latitude", "longitude"]
)

# Get structural columns vs all columns
nlp_feature_names = [
    "vader_compound_mean", "vader_compound_std", "vader_compound_min", 
    "pct_positive", "pct_negative", "review_length_mean", "review_length_std", 
    "review_length_max", "review_word_count_mean", "review_span_days", 
    "avg_days_between_reviews", "days_since_last_review", "dominant_topic"
] + [f"asp_{asp}_mean" for asp in ASPECTS] + [f"topic_{i}" for i in range(5)]

# Keep indexes of columns in processing matrix
all_features = cols_nlp
structural_indices = [i for i, name in enumerate(all_features) if name not in nlp_feature_names]

print(f"Total features in dataset: {len(all_features)}")
print(f"Structural features count: {len(structural_indices)}")

# Train baseline XGBoost on Structural Only
xgb_regr_structural = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_regr_structural.fit(X_train_nlp[:, structural_indices], y_train_nlp)
y_pred_struct = xgb_regr_structural.predict(X_val_nlp[:, structural_indices])
rmse_struct = np.sqrt(mean_squared_error(y_val_nlp, y_pred_struct))
r2_struct = r2_score(y_val_nlp, y_pred_struct)

# Train baseline XGBoost on Full Features (with NLP)
xgb_regr_full = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
xgb_regr_full.fit(X_train_nlp, y_train_nlp)
y_pred_full = xgb_regr_full.predict(X_val_nlp)
rmse_full = np.sqrt(mean_squared_error(y_val_nlp, y_pred_full))
r2_full = r2_score(y_val_nlp, y_pred_full)

print("=== Downstream Ablation Study (Task A - Price Prediction) ===")
print(f"Structural Only -> RMSE: {rmse_struct:.4f} | R2: {r2_struct:.4f}")
print(f"With NLP Features -> RMSE: {rmse_full:.4f} | R2: {r2_full:.4f}")
print(f"Delta -> R2 Lift: {r2_full - r2_struct:+.4f}")

In [ ]:
# Phase 7: Wrapper-Based Topic Optimization (NMF Tuning for maximum downstream R2)
best_r2_opt = -np.inf
best_params = {}

print("Running Downstream Wrapper-Based NMF Parameter Optimization...")
active_lids = consolidated_df.index.tolist()
active_texts = [listing_texts.get(lid, "") for lid in active_lids]

# Small NMF Parameter Grid to avoid long execution times
param_grid = [
    {"n_topics": 3, "max_features": 3000},
    {"n_topics": 5, "max_features": 5000}
]

for params in param_grid:
    # Extract temporary topic features
    temp_tfidf = TfidfVectorizer(max_features=params["max_features"], stop_words=COMBINED_STOPS, ngram_range=(1, 1), min_df=20)
    temp_tfidf_matrix = temp_tfidf.fit_transform(active_texts)
    
    temp_nmf = NMF(n_components=params["n_topics"], random_state=42, max_iter=200, init="nndsvda")
    temp_topic_matrix = temp_nmf.fit_transform(temp_tfidf_matrix)
    
    # Join and evaluate downstream
    temp_topic_df = pd.DataFrame(temp_topic_matrix, index=active_lids, columns=[f"temp_topic_{i}" for i in range(params["n_topics"])])
    temp_listings = lst_for_price_preproc.merge(temp_topic_df, left_on="id", right_index=True, how="left").fillna(0)
    
    X_t, X_v, y_t, y_v, c_t, _ = preprocess_data(temp_listings, cols_to_drop=["neighbourhood", "latitude", "longitude"])
    
    xgb_test = xgb.XGBRegressor(n_estimators=50, max_depth=4, learning_rate=0.1, random_state=42)
    xgb_test.fit(X_t, y_t)
    score_r2 = r2_score(y_v, xgb_test.predict(X_v))
    
    print(f"NMF Config (Topics: {params['n_topics']}, Features: {params['max_features']}) -> Downstream validation R2: {score_r2:.4f}")
    if score_r2 > best_r2_opt:
        best_r2_opt = score_r2
        best_params = params

print(f"Wrapper NMF Search Complete. Best Config: {best_params} with Downstream R2: {best_r2_opt:.4f}")


## 3. Preprocessing

#### 3.1 Preprocessing per Price Regression

Passiamo alla fase di preprocessing per il task di regressione su `price`. In questa fase, è necessario preparare i dati in modo che siano adatti per l'addestramento del modello di regressione. I passi che seguono sono:


*Scrivi passi quando li fai*

Procediamo con il creare un nuovo DataFrame `lst_for_price_preproc` a partire da `listings`, che contiene solo le feature necessarie per la predizione del prezzo. Effettuiamo il filtraggio delle feature come fatto nella sezione precedente. Lo ripetiamo qui per maggiore chiarezza:

In [ ]:
def prepare_price_regression_data_base(listings=listings):
    lst_for_price_preproc = listings.copy()

    # Definizione delle feature da mantenere per il task di regressione su price
    filtered_features = [
        "id",
        "city",
        "neighbourhood_cleansed",
        "latitude",
        "longitude",
        "property_type",
        "room_type",
        "accommodates",
        "bathrooms",
        "bedrooms",
        "beds",
        "amenities",
        "price"
    ]

    # Filtraggio feature per il task di regressione su price
    lst_for_price_preproc = lst_for_price_preproc[filtered_features]

    lst_for_price_preproc = lst_for_price_preproc.rename(columns={"neighbourhood_cleansed": "neighbourhood"})

    # Trattamento della variabile price per trasformarla in un valore numerico
    lst_for_price_preproc["price"] = (
    lst_for_price_preproc["price"]
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )

    # Trattamento dei valori mancanti per le feature numeriche
    lst_for_price_preproc = lst_for_price_preproc.dropna(subset=["bedrooms", "beds"])
    lst_for_price_preproc["bathrooms"] = lst_for_price_preproc["bathrooms"].fillna(0)

    # Filtraggio outliers dati da palesi errori di immissione
    lst_for_price_preproc = lst_for_price_preproc[
        (lst_for_price_preproc["bathrooms"] < 20) &
        (lst_for_price_preproc["bedrooms"] < 22) &
        (lst_for_price_preproc["beds"] < 35)
    ]

    lst_for_price_preproc["price"] = lst_for_price_preproc["price"].clip(upper=10000, lower=5)
    lst_for_price_preproc["price"] = np.log1p(lst_for_price_preproc["price"])

    lst_for_price_preproc["amenities"] = lst_for_price_preproc["amenities"].apply(ast.literal_eval)

    # Creazione di nuove feature a partire da quelle esistenti
    lst_for_price_preproc["beds_per_person"] = lst_for_price_preproc["beds"] / lst_for_price_preproc["accommodates"]
    lst_for_price_preproc["bedrooms_per_person"] = lst_for_price_preproc["bedrooms"] / lst_for_price_preproc["accommodates"]
    lst_for_price_preproc["bathrooms_per_person"] = lst_for_price_preproc["bathrooms"] / lst_for_price_preproc["accommodates"]
    lst_for_price_preproc["beds_per_bedroom"] = lst_for_price_preproc["beds"] / lst_for_price_preproc["bedrooms"]

    # Trattamento di inf e -inf derivanti da divisioni per zero o valori nulli
    cols = ["beds_per_person", "bedrooms_per_person", "bathrooms_per_person", "beds_per_bedroom"]

    lst_for_price_preproc[cols] = (
        lst_for_price_preproc[cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    return lst_for_price_preproc

lst_for_price_preproc = prepare_price_regression_data_base()

Guardiamo ora il risultato del preprocessing:

In [ ]:
lst_for_price_preproc.head()

Andiamo ora a modificare le categorie di `property_type` in modo da avere solo i top 10 più frequenti, e gli altri valori raggruppati in una categoria "other". In questo modo, si riduce la dimensionalità della variabile categorica, mantenendo comunque le categorie più rappresentative per la predizione del prezzo.

Creiamo una funzione keep_top_categories che prende in input una colonna categorica, il numero di categorie da mantenere e il nome di una categoria per le rimanenti.

In [ ]:
def keep_top_categories(df, column, top_n=10, other_label="Other"):
    top_values = df[column].value_counts().head(top_n).index
    df[column] = df[column].where(df[column].isin(top_values), other_label)
    return df, top_values

In [ ]:
lst_for_price_preproc, _ = keep_top_categories(
    lst_for_price_preproc,
    "property_type",
    top_n=10
)

lst_for_price_preproc.head()

Creiamo una variabile neighbourhood_price_median che rappresenta la mediana dei prezzi per ogni quartiere, in modo da avere una variabile numerica che rappresenta il prezzo mediano per quartiere.

In [ ]:
# lst_for_price_preproc['neighbourhood_price_median'] = lst_for_price_preproc.groupby('neighbourhood')['price'].transform('median')

Facciamo la stessa cosa per `neighbourhood`, mantenendo solo i top 50 quartieri più frequenti e raggruppando gli altri in "other". In questo modo, si riduce la dimensionalità della variabile categorica, mantenendo comunque i quartieri più rappresentativi per la predizione del prezzo.

In [ ]:
lst_for_price_preproc, _ = keep_top_categories(
    lst_for_price_preproc,
    "neighbourhood",
    top_n=50
)
lst_for_price_preproc.head()

Per le amenities, si è deciso di mantenere solo i 20 servizi più frequenti, e di raggruppare gli altri in una categoria "other". In questo modo, si riduce la dimensionalità della variabile categorica. Per ottenere un altra informazione utile, si è deciso di creare una nuova variabile numerica `n_amenities`, che rappresenta il numero di servizi offerti da ogni alloggio. Questa variabile potrebbe essere utile per la predizione del prezzo, in quanto un alloggio con più servizi potrebbe essere associato a un prezzo più elevato.
Facciamo diventare amenities un set in modo da non ripetere più volte lo stesso servizio.

In [ ]:
def preprocess_amenities(df, top_n=30, other_label="Other"):
    df = df.copy()
    df["n_amenities"] = df["amenities"].apply(len)

    top_amenities = (
        df["amenities"]
        .explode()
        .value_counts()
        .sort_values(ascending=False)
        .head(top_n)
        .index
    )

    df["amenities"] = df["amenities"].apply(
        lambda x: list({a if a in top_amenities else other_label for a in x})
    )

    return df, top_amenities

lst_for_price_preproc, _ = preprocess_amenities(lst_for_price_preproc)

In [ ]:
lst_for_price_preproc.head()

Un altro dato utile da estrarre potrebbe essere il calcolo della distanza di ogni alloggio dal centro della città, in modo da avere una variabile numerica che rappresenta la posizione geografica dell'alloggio. Per fare ciò, bisogna calcolare la distanza tra le coordinate geografiche di ogni alloggio e le coordinate del centro della città. In questo modo, si ottiene una nuova variabile numerica `distance_from_city_center` che potrebbe essere utile per la predizione del prezzo, siccome gli alloggi più vicini al centro potrebbero essere associati a un prezzo più elevato.

Creiamo quindi un dizionario con le coordinate del centro di ogni città presente nel dataset, in modo da poter calcolare la distanza di ogni alloggio dal centro della città. I dati sono stati presi da un LLM, e sono approssimativi, ma dovrebbero essere sufficienti per il nostro scopo.

In [ ]:
city_centers = {
    "bergamo": [{"lat": 45.6983, "lon": 9.6773}],
    "bologna": [{"lat": 44.4949, "lon": 11.3426}],
    "firenze": [{"lat": 43.7696, "lon": 11.2558}],
    "milano":  [{"lat": 45.4654, "lon": 9.1859}],
    "napoli":  [{"lat": 40.8518, "lon": 14.2681}],
    "puglia": [
        {"city": "Bari",      "lat": 41.1171, "lon": 16.8719},
        {"city": "Foggia",    "lat": 41.4621, "lon": 15.5444},
        {"city": "Taranto",   "lat": 40.4644, "lon": 17.2470},
        {"city": "Lecce",     "lat": 40.3516, "lon": 18.1752},
        {"city": "Brindisi",  "lat": 40.6326, "lon": 17.9417},
        {"city": "Andria",    "lat": 41.2278, "lon": 16.2961},
        {"city": "Barletta",  "lat": 41.3197, "lon": 16.2820},
    ],
    "roma":    [{"lat": 41.9028, "lon": 12.4964}],
    "sicilia": [
        {"city": "Palermo",   "lat": 38.1157, "lon": 13.3615},
        {"city": "Catania",   "lat": 37.5079, "lon": 15.0830},
        {"city": "Messina",   "lat": 38.1938, "lon": 15.5540},
        {"city": "Siracusa",  "lat": 37.0755, "lon": 15.2866},
        {"city": "Agrigento", "lat": 37.3111, "lon": 13.5765},
        {"city": "Trapani",   "lat": 38.0176, "lon": 12.5365},
        {"city": "Ragusa",    "lat": 36.9282, "lon": 14.7256},
    ],
    "trentino": [
        {"city": "Trento",      "lat": 46.0748, "lon": 11.1217},
        {"city": "Bolzano",     "lat": 46.4983, "lon": 11.3548},
        {"city": "Rovereto",    "lat": 45.8912, "lon": 11.0396},
        {"city": "Merano",      "lat": 46.6713, "lon": 11.1594},
        {"city": "Bressanone",  "lat": 46.7154, "lon": 11.6565},
    ],
    "venezia": [{"lat": 45.4408, "lon": 12.3155}],
}

Per il centro delle regioni sono stati scelti i centri delle città più importanti. Ora creiamo una funzione `calculate_distances_from_city_center` che calcola la distanza tra le coordinate di ogni alloggio e le coordinate del centro della città. Siccome le regioni hanno più città, facciamo in modo che ritorni la lista delle distanze da ogni centro, in modo da poter prendere la distanza minima come distanza dell'alloggio dal centro della regione.

Siccome la terra è sferica, è necessario utilizzare la formula dell'haversine per calcolare la distanza tra due punti sulla superficie terrestre. L'idea della formula è quella di, dati due punti su una sfera, calcolare la lunghezza dell'arco che li collega passando per la sua superficie. 

Quindi:
1. Convertiamo le coordinate da gradi a radianti.
2. Calcoliamo le differenze di latitudine e longitudine.
3. Applichiamo la formula dell'haversine per ottenere la distanza in chilometri.

La formula dell'haversine è la seguente:

$$a = sin(\frac{dlat}{2})^2 + cos(lat1) * cos(lat2) * sin(\frac{dlon}{2})^2$$
$$c = 2 * atan2(\sqrt{a}, \sqrt{1−a})$$
$$d = R * c$$
Dove:
- `dlat` è la differenza di latitudine in radianti.
- `dlon` è la differenza di longitudine in radianti.
- `R` è il raggio della Terra (circa 6371 km).

Otteniamo alla fine $d$, che rappresenta la distanza in chilometri tra l'alloggio e il centro della città.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # raggio della Terra in metri
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

In [ ]:
def calculate_distances_from_city_center(city, lat, lon, city_centers=city_centers):
    centers = city_centers.get(city, [])
    if not centers:
        return None
    distances = [haversine(lat, lon, c["lat"], c["lon"]) for c in centers]
    return distances if distances else None

Applichiamo ora il metodo apply alle colonne `city`, `latitude`, e `longitude` per calcolare la distanza di ogni alloggio dal centro della città, e creiamo la nuova colonna.

In [ ]:
lst_for_price_preproc["distance_from_city_center"] = lst_for_price_preproc.apply(
    lambda row: min(calculate_distances_from_city_center(row["city"], row["latitude"], row["longitude"])),
    axis=1
)

In [ ]:
lst_for_price_preproc.head()

Tuttavia potrebbero esserci anche degli altri punti di interesse, come ad esempio i punti di interesse turistico, che potrebbero essere utili per la predizione del prezzo. Per questo motivo, si potrebbe considerare di calcolare anche la distanza di ogni alloggio da questi punti di interesse, e creare delle nuove variabili numeriche che rappresentano queste distanze. Estraiamo sempre da un LLM le coordinate di alcuni punti di interesse turistico per le città e le regioni presenti nel dataset:

In [ ]:
poi_by_city = {
    "bergamo": [
        {"lat": 45.7040, "lon": 9.6623}, # Campanone
        {"lat": 45.7035, "lon": 9.6622}, # Cappella Colleoni
        {"lat": 45.7035, "lon": 9.6627}, # Cattedrale di Bergamo
        {"lat": 45.7034, "lon": 9.6623}, # Basilica di Santa Maria Maggiore
        {"lat": 45.7038, "lon": 9.6627}, # Palazzo della Ragione
        {"lat": 45.7041, "lon": 9.6630}, # Fontana Contarini
        {"lat": 45.7041, "lon": 9.6578}, # Casematte di San Giovanni
    ],
    "bologna": [
        {"lat": 44.4942, "lon": 11.3467}, # Le Due Torri (Asinelli e Garisenda)
        {"lat": 44.4938, "lon": 11.3431}, # Piazza Maggiore
        {"lat": 44.4919, "lon": 11.3433}, # Palazzo dell'Archiginnasio
        {"lat": 44.4911, "lon": 11.3436}, # Portici di Bologna
        {"lat": 44.4923, "lon": 11.3483}, # Piazza Santo Stefano
        {"lat": 44.4937, "lon": 11.3422}, # Torre dell'Orologio
        {"lat": 44.4896, "lon": 11.3440}, # Basilica di San Domenico
    ],
    "firenze": [
        {"lat": 43.7693, "lon": 11.2562}, # Palazzo Vecchio
        {"lat": 43.7731, "lon": 11.2560}, # Cattedrale di Santa Maria del Fiore
        {"lat": 43.7697, "lon": 11.2556}, # Piazza della Signoria
        {"lat": 43.7686, "lon": 11.2623}, # Basilica di Santa Croce
        {"lat": 43.7629, "lon": 11.2651}, # Piazzale Michelangelo
        {"lat": 43.7729, "lon": 11.2558}, # Campanile di Giotto
        {"lat": 43.7692, "lon": 11.2555}, # Loggia dei Lanzi
    ],
    "milano": [
        {"lat": 45.4641, "lon": 9.1919}, # Duomo di Milano
        {"lat": 45.4642, "lon": 9.1897}, # Piazza del Duomo
        {"lat": 45.4705, "lon": 9.1793}, # Castello Sforzesco
        {"lat": 45.4658, "lon": 9.1899}, # Galleria Vittorio Emanuele II
        {"lat": 45.4660, "lon": 9.1710}, # Basilica di Santa Maria delle Grazie
        {"lat": 45.4864, "lon": 9.1826}, # Torre Arcobaleno
        {"lat": 45.4692, "lon": 9.1809}, # Piazza Castello (Fontana)
    ],
    "napoli": [
        {"lat": 40.8651, "lon": 14.2474}, # Catacombe di San Gennaro
        {"lat": 40.8373, "lon": 14.2455}, # Napoli Sotterranea
        {"lat": 40.8362, "lon": 14.2494}, # Palazzo Reale di Napoli
        {"lat": 40.8536, "lon": 14.2505}, # Museo Archeologico Nazionale
        {"lat": 40.8385, "lon": 14.2527}, # Castel Nuovo (Maschio Angioino)
        {"lat": 40.8358, "lon": 14.2486}, # Piazza del Plebiscito
        {"lat": 40.8436, "lon": 14.2408}, # Certosa e Museo di San Martino
        {"lat": 40.8595, "lon": 14.2485}, # Catacombe di San Gaudioso
    ],
    "puglia": [
        {"lat": 41.1303, "lon": 16.8701}, # Basilica di San Nicola (Bari)
        {"lat": 41.1286, "lon": 16.8688}, # Cattedrale di San Sabino (Bari)
        {"lat": 41.1279, "lon": 16.8664}, # Castello Svevo di Bari
        {"lat": 40.8759, "lon": 17.1480}, # Grotte di Castellana
        {"lat": 41.0848, "lon": 16.2709}, # Castel del Monte
    ],
    "roma": [
        {"lat": 41.8921, "lon": 12.4864}, # Foro Romano
        {"lat": 41.9009, "lon": 12.4833}, # Fontana di Trevi
        {"lat": 41.8986, "lon": 12.4769}, # Pantheon
        {"lat": 41.8902, "lon": 12.4922}, # Colosseo
        {"lat": 41.8992, "lon": 12.4731}, # Piazza Navona
        {"lat": 41.9107, "lon": 12.4764}, # Piazza del Popolo
        {"lat": 41.8956, "lon": 12.4722}, # Campo de' Fiori
        {"lat": 41.9060, "lon": 12.4828}, # Scalinata di Trinità dei Monti
    ],
    "sicilia": [
        {"lat": 38.1132, "lon": 13.3530}, # Cattedrale di Palermo
        {"lat": 38.1109, "lon": 13.3517}, # Palazzo dei Normanni
        {"lat": 37.5025, "lon": 15.0871}, # Fontana dell'Elefante (Catania)
        {"lat": 37.5024, "lon": 15.0877}, # Basilica di Sant'Agata (Catania)
        {"lat": 37.5024, "lon": 15.0906}, # Palazzo Biscari (Catania)
        {"lat": 37.5090, "lon": 15.1025}, # Museo Sbarco in Sicilia 1943
        {"lat": 37.5022, "lon": 15.0877}, # Terme Achilliane (Catania)
    ],
    "trentino": [
        {"lat": 46.0715, "lon": 11.1271}, # Castello del Buonconsiglio
        {"lat": 46.0673, "lon": 11.1215}, # Piazza del Duomo (Trento)
        {"lat": 46.0695, "lon": 11.1214}, # Torre Mirana
        {"lat": 46.4983, "lon": 11.3548}, # Piazza Walther (Bolzano)
        {"lat": 46.5011, "lon": 11.3570}, # Museo di Scienze Naturali Alto Adige
        {"lat": 46.4828, "lon": 11.1322}, # Cascata di Tret
    ],
    "venezia": [
        {"lat": 45.4346, "lon": 12.3397}, # Basilica di San Marco
        {"lat": 45.4337, "lon": 12.3404}, # Palazzo Ducale
        {"lat": 45.4380, "lon": 12.3359}, # Ponte di Rialto
        {"lat": 45.4340, "lon": 12.3409}, # Ponte dei Sospiri
        {"lat": 45.4342, "lon": 12.3385}, # Piazza San Marco
        {"lat": 45.4340, "lon": 12.3390}, # Campanile di San Marco
        {"lat": 45.4348, "lon": 12.3368}, # Gondola Ride Experience
    ],
}

Da queste coordinate si possono ricavare delle nuove variabili numeriche che rappresentano la distanza di ogni alloggio da il punto di interesse più vicino, utilizzando la stessa funzione `calculate_distances_from_city_center` che abbiamo creato prima, ma adattandola per calcolare la distanza da un punto di interesse invece che dal centro della città.

Altre feature interessanti da estrarre potrebbero essere il numero di punti di interesse a meno di 250m, 500m, 1km, 2km, 5km. In questo modo, si ottengono nuove variabili numeriche che rappresentano la vicinanza di ogni alloggio a punti di interesse turistico, che potrebbero essere utili per la predizione del prezzo, in quanto gli alloggi più vicini a punti di interesse potrebbero essere associati a un prezzo più elevato. Creiamo quindi una nuova funzione `calculate_n_poi` che calcola il numero di punti di interesse a distanze variabili.

In [ ]:
def calculate_n_poi(city, lat, lon, thresh):
    distances = calculate_distances_from_city_center(city, lat, lon)
    if not distances:
        return None
    count = len([distance for distance in distances if distance <= thresh])
    return count

In [ ]:
distances_from_poi = lst_for_price_preproc.apply(
    lambda row: calculate_distances_from_city_center(row["city"], row["latitude"], row["longitude"], city_centers=poi_by_city),
    axis=1
)

print(distances_from_poi.head())

lst_for_price_preproc["distance_from_poi"] = distances_from_poi.apply(lambda dists: min(dists))

n_poi_250 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 250))

n_poi_500 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 500))

n_poi_1000 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 1000))

n_poi_2000 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 2000))

n_poi_5000 = distances_from_poi.apply(lambda dists: sum(1 for d in dists if d <= 5000))

Queste variabili sono fortemente correlate tra loro, il che potrebbe causare problemi di multicollinearità nel modello di regressione. Per questo motivo si potrebbe creare una nuova variabile che rappresenta la densità di punti di interesse, calcolata come media pesata del numero di punti di interesse a diverse distanze, in modo da ridurre la dimensionalità del dataset e migliorare la performance del modello di regressione. La formula per calcolare la densità di punti di interesse potrebbe essere la seguente:

$$poi\_density = n\_poi\_250m * w1 + n\_poi\_500m * w2 + n\_poi\_1km * w3 + n\_poi\_2km * w4 + n\_poi\_5km * w5$$
Dove `w1`, `w2`, `w3`, `w4`, e `w5` sono i pesi che rappresentano l'importanza relativa dei punti di interesse a diverse distanze.

Assegnamo quindi dei pesi decrescenti ai punti di interesse a distanze , la cui somma è 1, ovvero:
- `w1` = 0.25 (peso per i punti di interesse a 250 metri)
- `w2` = 0.25 (peso per i punti di interesse a 500 metri)
- `w3` = 0.25 (peso per i punti di interesse a 1 chilometro)
- `w4` = 0.15 (peso per i punti di interesse a 2 chilometri)
- `w5` = 0.1 (peso per i punti di interesse a 5 chilometri)

In [ ]:
lst_for_price_preproc["poi_density"] = n_poi_250 * 0.25 + n_poi_500 * 0.25 + n_poi_1000 * 0.25 + n_poi_2000 * 0.15 + n_poi_5000 * 0.1

In [ ]:
lst_for_price_preproc.head()

In [ ]:
def calculate_distances_from_points(lat, lon, points):
    if not points:
        return []

    return [
        haversine(lat, lon, p["lat"], p["lon"])
        for p in points
    ]


def process_poi_distances(df, poi_dict=poi_by_city, thresholds=(250, 500, 1000, 2000, 5000)):
    df = df.copy()

    distances = df.apply(
        lambda row: calculate_distances_from_points(
            row["latitude"],
            row["longitude"],
            poi_dict.get(row["city"], [])
        ),
        axis=1
    )

    df["distance_from_poi"] = distances.apply(lambda dists: min(dists) if dists else np.nan)

    weights = {
        250: 0.25,
        500: 0.25,
        1000: 0.25,
        2000: 0.15,
        5000: 0.10
    }
    df["poi_density"] = distances.apply(
        lambda dists: sum(
            sum(d <= th for d in dists) * weight
            for th, weight in weights.items()
        ) if dists else 0
    )

    return df


lst_for_price_preproc = process_poi_distances(lst_for_price_preproc)

In [ ]:
def build_price_preprocessing_dataframe(
        listings_df=listings,
        top_n_property_types=10,
        top_n_neighbourhoods=50,
        top_n_amenities=30
    ):
    df = prepare_price_regression_data_base(listings_df)

    df, _ = keep_top_categories(df, "property_type", top_n=top_n_property_types, other_label="Other")
    # df, _ = keep_top_categories(df, "neighbourhood", top_n=top_n_neighbourhoods, other_label="Other")

    df, _ = preprocess_amenities(df, top_n=top_n_amenities, other_label="Other")

    df = process_poi_distances(df)

    return df

lst_for_price_preproc = build_price_preprocessing_dataframe(listings)
lst_for_price_preproc.head()

Procediamo ora con il preprocessing delle variabili categoriche `neighbourhood_cleansed`, `property_type`, `room_type` e `amenities`. Per fare ciò, è necessario trasformare queste variabili in nuove variabili binarie, in modo da poterle utilizzare per l'addestramento del modello di regressione. Usiamo i metodi di scikit-learn `OneHotEncoder` e `MultiLabelBinarizer` per trasformare le variabili categoriche in variabili binarie.

Tuttavia è necessario creare una classe custom MLBTransformer che utilizza `MultiLabelBinarizer` per trasformare la variabile `amenities`, in quanto questa variabile contiene una lista di servizi per ogni record, e non è possibile utilizzare direttamente MultiLabelBinarizer per trasformarla in variabili binarie.

Questa classe custom `MLBTransformer` implementa i metodi `fit`, `transform` e `fit_transform` per adattarsi alle interfacce BaseEstimator e TransformerMixin di scikit-learn. Il metodo `fit` adatta il `MultiLabelBinarizer` ai dati, il metodo `transform` trasforma i dati in un array binario, e il metodo `fit_transform` combina entrambe le operazioni. Inoltre si vuole mantenere i nomi delle feature originali, quindi si aggiunge un metodo `get_feature_names_out` che restituisce i nomi delle feature trasformate (che in MultiLabelBinarizer sono nell'attributo `classes_`).

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

class MLBTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer(sparse_output=False)

    def fit(self, X, y=None):
        self.mlb.fit(X)
        return self

    def transform(self, X):
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return np.array([c for c in self.mlb.classes_])

In [ ]:
from sklearn.preprocessing import TargetEncoder

def preprocess_data_w_target_encoder(listings, cols_to_drop, seed=7112004):
    categorical_cols = ["city", "property_type", "room_type"]

    # Aggiungo "neighbourhood" alle colonne da codificare con TargetEncoder
    target_enc_cols = ["neighbourhood"]
    amenities_col = "amenities"

    ohe = OneHotEncoder(sparse_output=False, drop="first")

    # Il TargetEncoder nativo applica lo smoothing in modo automatico
    # cv=5 serve per evitare il data leakage durante il calcolo delle medie sul train set
    te = TargetEncoder(smooth="auto", cv=5, random_state=seed)

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", ohe, categorical_cols),
            ("target_geo", te, target_enc_cols),
            ("amenities", MLBTransformer(), amenities_col)
        ],
        remainder="passthrough"
    )

    X = listings.drop(columns=["id", "price"] + cols_to_drop)
    y = listings["price"]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=1/3, random_state=seed)

    # Passiamo anche y_train al fit del preprocessor per permettere al TargetEncoder di calcolare le medie corrette
    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_val_processed = preprocessor.transform(X_val)

    cols = X_train.columns.tolist() + preprocessor.get_feature_names_out().tolist()
    return X_train_processed, X_val_processed, y_train, y_val, cols, preprocessor

In [ ]:
X_train, X_val, y_train, y_val, cols, preprocessor = preprocess_data_w_target_encoder(lst_for_price_preproc, cols_to_drop=["latitude", "longitude"])

#### 3.2 Preprocessing per Occupancy Regression

Passiamo alla fase di preprocessing per il task di regressione su `occupancy`. Anche in questa fase, è necessario preparare i dati in modo che siano adatti per l'addestramento del modello di regressione. 
I passi sono molto simili a quelli del task di regressione su `price`, con alcune differenze dovute alla natura del target duqneu partiremo creando una copia del DataFrame `lst_for_price_preproc` e lo chiameremo `lst_for_occupancy_preproc`, in modo da avere un nuovo DataFrame su cui lavorare, senza modificare quello creato per il task di regressione su `price`.

In [ ]:
lst_for_occupancy_preproc = lst_for_price_preproc.copy()
print(lst_for_occupancy_preproc.shape)
lst_for_occupancy_preproc.head()

L'obiettvo adesso è quellodi adattare lst_for_occupancy_preproc per il task B (Occupancy Regression) aggiungendo le feature più rilevanti per la predizione del tasso di occupazione, e rimuovendo le feature che potrebbero creare leakage, come `price`, `estimated_revenue_l365d`, tutte le colonne `availability_*` e le colonne `number_of_reviews*`.

In [ ]:
# A) Pulizia righe e target (dedup robusta anche con colonne list-type)
_dedup_cols = [c for c in lst_for_occupancy_preproc.columns if c != "amenities"]
lst_for_occupancy_preproc = lst_for_occupancy_preproc.drop_duplicates(subset=_dedup_cols).copy()

if "estimated_occupancy_l365d" in lst_for_occupancy_preproc.columns:
    lst_for_occupancy_preproc = lst_for_occupancy_preproc.dropna(subset=["estimated_occupancy_l365d"]).reset_index(drop=True)

# B) Rimozione colonne a leakage
leakage_exact = {"price", "estimated_revenue_l365d"}
leakage_prefixes = ("availability_", "number_of_reviews")

cols_to_drop = [
    c for c in lst_for_occupancy_preproc.columns
    if c in leakage_exact or any(c.startswith(p) for p in leakage_prefixes)
]

if cols_to_drop:
    lst_for_occupancy_preproc = lst_for_occupancy_preproc.drop(columns=cols_to_drop)

Effettiamo qunindi il parsing di amenities e la creazione feature numerica di supporto prima del merge con calendar, in modo da avere questa feature anche in caso di merge fallito per assenza di calendar

In [ ]:
if "amenities" in lst_for_occupancy_preproc.columns:
    def _safe_parse_amenities(x):
        if isinstance(x, list):
            return x
        if pd.isna(x):
            return []
        if isinstance(x, str):
            try:
                parsed = ast.literal_eval(x)
                return parsed if isinstance(parsed, list) else []
            except (ValueError, SyntaxError):
                return []
        return []

    lst_for_occupancy_preproc["amenities"] = lst_for_occupancy_preproc["amenities"].apply(_safe_parse_amenities)
    lst_for_occupancy_preproc["n_amenities"] = lst_for_occupancy_preproc["amenities"].apply(len)

Eseguiamo il merge della struttura dati con i dati iposrati da calendar, in modo da avere a disposizione le feature di occupazione per ogni alloggio. Il merge viene effettuato sulla colonna `id`, che rappresenta l'identificativo univoco di ogni alloggio.

In [ ]:
# A) Merge con feature calendar aggregate per listing (se calendars e' disponibile)
if "calendars" in globals() and isinstance(calendars, pd.DataFrame) and not calendars.empty:
    calendar_key = next((k for k in ["listing_id", "id"] if k in calendars.columns), None)
    calendar_feature_candidates = [
        "monthly_occupancy_mean", "occupancy_rate", "total_occ_days", "total_observed_days"
    ]
    calendar_feature_candidates += [f"occupancy_rate_{m:02d}" for m in range(1, 13)]
    calendar_feature_candidates += [f"occ_days_{m:02d}" for m in range(1, 13)]

    calendar_features = [c for c in calendar_feature_candidates if c in calendars.columns]

    if calendar_key is not None and calendar_features and "id" in lst_for_occupancy_preproc.columns:
        cal_agg = calendars[[calendar_key] + calendar_features].copy()
        cal_agg = cal_agg.groupby(calendar_key, as_index=False).mean(numeric_only=True)
        if calendar_key != "id":
            cal_agg = cal_agg.rename(columns={calendar_key: "id"})

        lst_for_occupancy_preproc = lst_for_occupancy_preproc.merge(cal_agg, on="id", how="left")

        # Imputazione semplice delle feature calendar mancanti con mediana
        for c in calendar_features:
            if c in lst_for_occupancy_preproc.columns:
                lst_for_occupancy_preproc[c] = lst_for_occupancy_preproc[c].fillna(lst_for_occupancy_preproc[c].median())
else:
    print("Calendars non disponibile nel kernel: merge calendar saltato.")

# B) Riduzione cardinalita' di property_type (utile per encoding successivo)
if "property_type" in lst_for_occupancy_preproc.columns:
    top_prop = lst_for_occupancy_preproc["property_type"].value_counts().head(15).index
    lst_for_occupancy_preproc["property_type"] = lst_for_occupancy_preproc["property_type"].where(
        lst_for_occupancy_preproc["property_type"].isin(top_prop),
        "Other"
    )

E infine stampiamo i risultati del preprocessing:

In [ ]:
print("Shape finale lst_for_occupancy_preproc:", lst_for_occupancy_preproc.shape)
print("Colonne finali:", len(lst_for_occupancy_preproc.columns))

display(lst_for_occupancy_preproc.head())
display(lst_for_occupancy_preproc.describe().T)

A questo punto quello che ci resta da fare è effettuare il preprocessing delle variabili categoriche `neighbourhood_cleansed`, `property_type`, `room_type` e `amenities`, in modo simile a quanto fatto per il task di regressione su `price`. Per fare ciò, è necessario trasformare queste variabili in nuove variabili binarie, in modo da poterle utilizzare per l'addestramento del modello di regressione. Usiamo i metodi di scikit-learn `OneHotEncoder` e `MultiLabelBinarizer` per trasformare le variabili categoriche in variabili binarie.

Otterremo dunque i Train set e il Validation set per il task di regressione su `occupancy`, che conterranno le feature numeriche e le variabili binarie ottenute dal preprocessing delle variabili categoriche, pronte per essere utilizzate per l'addestramento del modello di regressione.

In [ ]:
# Merge target occupancy if it is missing in the current dataframe
if "estimated_occupancy_l365d" not in lst_for_occupancy_preproc.columns and "estimated_occupancy_l365d" in listings.columns:
    target_from_listings = listings[["id", "estimated_occupancy_l365d"]].drop_duplicates(subset=["id"])
    lst_for_occupancy_preproc = lst_for_occupancy_preproc.merge(target_from_listings, on="id", how="left")


def preprocess_occupancy_data(listings, cols_to_drop, seed=7112004):
    categorical_cols = ["city", "property_type", "room_type"]
    amenities_col = "amenities"

    ohe = OneHotEncoder(sparse_output=False, drop="first")

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", ohe, categorical_cols),
            ("amenities", MLBTransformer(), amenities_col),
        ],
        remainder="passthrough",
    )

    drop_cols = ["id", "estimated_occupancy_l365d"] + list(cols_to_drop)
    drop_cols = [c for c in drop_cols if c in listings.columns]

    X = listings.drop(columns=drop_cols)
    y = listings["estimated_occupancy_l365d"]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=1 / 3, random_state=seed)
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)
    cols = preprocessor.get_feature_names_out().tolist()

    return X_train_processed, X_val_processed, y_train, y_val, cols, preprocessor


X_train_occ, X_val_occ, y_train_occ, y_val_occ, cols_occ, preprocessor_occ = preprocess_occupancy_data(
    lst_for_occupancy_preproc,
    cols_to_drop=["neighbourhood", "latitude", "longitude"],
)

print("X_train_occ:", X_train_occ.shape)
print("X_val_occ:", X_val_occ.shape)
print("y_train_occ:", y_train_occ.shape)
print("y_val_occ:", y_val_occ.shape)


## 4. Addestramento e Valutazione dei Modelli

### 4.1 Price Regression

Partiamo ora con l'addestramento e la valutazione dei modelli per il task di regressione su `price`. In questa fase, è necessario utilizzare il dataset preprocessato per addestrare un modello di regressione che possa predire il prezzo degli alloggi sulla base delle feature disponibili. I passi che seguono sono:

**Scrivi passi quando li fai**

#### Valutazione dei modelli

Per valutare la performance del modello di regressione, è necessario utilizzare delle metriche di valutazione appropriate. Le metriche più comuni per la regressione sono:
- **MSE (Mean Squared Error)**: misura l'errore medio quadratico tra le predizioni del modello e i valori reali. Un valore più basso indica una migliore performance.
- **Relative Error (RE)**: misura l'errore relativo tra le predizioni del modello e i valori reali. Un valore più basso indica una migliore performance.
- **R² (R-squared)**: misura la proporzione della varianza nei dati che è spiegata dal modello. Un valore più alto indica una migliore performance, con 1 che rappresenta una perfetta predizione.

Definiamo le funzioni per calcolare queste metriche di valutazione:

In [ ]:
from sklearn.metrics import mean_squared_error

def relative_error(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true))

def print_eval(X, y, model):
    print("   Mean squared error: {:.5}".format(mean_squared_error(model.predict(X), y)))
    print("       Relative error: {:.5%}".format(relative_error(model.predict(X), y)))
    print("R-squared coefficient: {:.5}".format(model.score(X, y)))

#### Regressione Lasso

*Scrivi*

In [ ]:
# Regressione Lasso

from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ("scale", StandardScaler(with_mean=False)),
    ("regr", Lasso())
])

grid = {
    "regr__alpha": [0.0005, 0.001, 0.005]
}

# cols = ["rank_test_score","mean_test_score","mean_train_score","params"]
gs = GridSearchCV(model, param_grid=grid, cv=5)
gs.fit(X_train, y_train)

pd.DataFrame(gs.cv_results_).sort_values("mean_test_score", ascending=False)

In [ ]:
lst_for_price_preproc.info()

Come si può vedere il modello di regressione Lasso ha una performance piuttosto bassa, con un $R^2$ di 0.176. Questo è dovuto al fatto che abbiamo costruito un modello lineare, e quindi potrebbe non essere in grado di catturare le relazioni non lineari tra le feature e il prezzo.

In ogni caso possiamo guardare i coefficienti del modello a 0 per capire quali feature sono state considerate più importanti per la predizione del prezzo. Questo perchè la regolarizzazione L1 tende a azzerare i coefficienti delle feature meno importanti. Guardiamo i coefficienti a 0 con alpha=0.1:

In [ ]:
model = Pipeline([
    ("scale",  StandardScaler(with_mean=False)),
    ("regr", Lasso(alpha=0.0005))
])
model.fit(X_train, y_train);

In [ ]:
len(model.named_steps["regr"].coef_)

In [ ]:
lasso = pd.Series(model.named_steps["regr"].coef_, preprocessor.get_feature_names_out())
n_cols = lasso.count()
n_cols_zero = lasso[lasso != 0].count()
print(f"Numero di feature con coefficiente diverso da zero: {n_cols_zero} / {n_cols}")
print("Top feature più importanti (in valore assoluto):")
for feature, coef in lasso[lasso != 0].abs().sort_values(ascending=False).items():
    print(f"{feature}: {coef:.4f}")
print("Feature con coefficiente zero:")
for feature, coef in lasso[lasso == 0].items():
    print(f"{feature}: {coef:.4f}")

In [ ]:
print("Valutazione su training set:")
print_eval(X_train, y_train, model)
print("\nValutazione su validation set:")
print_eval(X_val, y_val, model)

I pesi che il modello di regressione Lasso ha assegnato alle feature Sono molto alti. Questo è un segnale che il modelo non riesce a catturare le relazioni tra le feature e il prezzo.

Probabilemente è necessario utilizzare un modello più complesso, come ad esempio un modello di regressione non lineare, o un modello di regressione con interazioni tra le feature, per migliorare la performance del modello di regressione.

Proviamo ora ad utilizzare un modello di regressione Ridge con feature polinomiali di diverso grado, per vedere se è in grado di catturare meglio le relazioni tra le feature e il prezzo, e quindi migliorare la performance del modello di regressione.

In [ ]:
# from sklearn.linear_model import Ridge
# from sklearn.preprocessing import PolynomialFeatures

# model = Pipeline([
#     ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
#     ('scaler', StandardScaler()),
#     ('ridge', Ridge())
# ])

# grid = {
#     'ridge__alpha': [0.1, 1, 10]
# }

# gs = GridSearchCV(model, param_grid=grid, cv=3)
# print(X_train)
# gs.fit(X_train, y_train)

# pd.DataFrame(gs.cv_results_).sort_values("mean_test_score", ascending=False)

In [ ]:
from sklearn.preprocessing import TargetEncoder

def preprocess_data_w_target_encoder(listings, cols_to_drop, seed=7112004):
    categorical_cols = ["city", "property_type", "room_type"]

    # Aggiungo "neighbourhood" alle colonne da codificare con TargetEncoder
    target_enc_cols = ["neighbourhood"]
    amenities_col = "amenities"

    ohe = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")

    # Il TargetEncoder nativo applica lo smoothing in modo automatico
    # cv=5 serve per evitare il data leakage durante il calcolo delle medie sul train set
    te = TargetEncoder(smooth="auto", cv=5, random_state=seed)

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", ohe, categorical_cols),
            ("target_geo", te, target_enc_cols),
            ("amenities", MLBTransformer(), amenities_col)
        ],
        remainder="passthrough"
    )

    X = listings.drop(columns=["id", "price"] + cols_to_drop)
    y = listings["price"]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=1/3, random_state=seed)

    # Passiamo anche y_train al fit del preprocessor per permettere al TargetEncoder di calcolare le medie corrette
    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_val_processed = preprocessor.transform(X_val)

    cols = X_train.columns.tolist() + preprocessor.get_feature_names_out().tolist()
    return X_train_processed, X_val_processed, y_train, y_val, cols, preprocessor

In [ ]:
X_train, X_val, y_train, y_val, cols, preprocessor = preprocess_data_w_target_encoder(lst_for_price_preproc, cols_to_drop=["latitude", "longitude"])

In [ ]:
gs.fit(X_train, y_train)

In [ ]:
pd.DataFrame(gs.cv_results_).sort_values("mean_test_score", ascending=False)

In [ ]:
def print_lasso_features_eval(model, preprocessor):
    lasso = pd.Series(model.named_steps["regr"].coef_, preprocessor.get_feature_names_out())
    n_cols = lasso.count()
    n_cols_zero = lasso[lasso != 0].count()
    print(f"Numero di feature con coefficiente diverso da zero: {n_cols_zero} / {n_cols}")
    print("Top feature più importanti (in valore assoluto):")
    for feature, coef in lasso[lasso != 0].abs().sort_values(ascending=False).items():
        print(f"{feature}: {coef:.4f}")
    print("Feature con coefficiente zero:")
    for feature, coef in lasso[lasso == 0].items():
        print(f"{feature}: {coef:.4f}")


In [ ]:
model = Pipeline([
    ("scale",  StandardScaler(with_mean=False)),
    ("regr", Lasso(alpha=0.0005))
])
model.fit(X_train, y_train);

In [ ]:
print_lasso_features_eval(model, preprocessor)

In [ ]:
def preprocess_data_with_scale(listings, cols_to_drop, seed=7112004):
    target_col = "price"
    id_col = "id"
    amenities_col = "amenities"
    target_enc_col = "neighbourhood"

    cols_to_remove = [id_col, target_col] + cols_to_drop

    X = listings.drop(columns=[c for c in cols_to_remove if c in listings.columns])
    y = listings[target_col]

    # Selezioniamo le colonne categoriche (escludendo quelle gestite da TargetEncoder e MLBTransformer)
    categorical_cols = [c for c in X.select_dtypes(include=["object", "str", "category"]).columns
                        if c != amenities_col and c != target_enc_col]

    # Selezioniamo tutte le colonne numeriche continue
    numeric_continuous_cols = X.select_dtypes(include=[np.number]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore"), categorical_cols),
            ("target_geo", TargetEncoder(smooth="auto", cv=5, random_state=seed), [target_enc_col] if target_enc_col in X.columns else []),
            ("amenities", MLBTransformer(), amenities_col),
            ("num_continuous", StandardScaler(with_mean=True, with_std=True), numeric_continuous_cols)
        ],
        remainder="drop"
    )

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=1/3, random_state=seed)

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_val_processed = preprocessor.transform(X_val)

    cols = preprocessor.get_feature_names_out().tolist()

    return X_train_processed, X_val_processed, y_train, y_val, cols, preprocessor

In [ ]:
X_train, X_val, y_train, y_val, cols, preprocessor = preprocess_data_w_target_encoder(lst_for_price_preproc, cols_to_drop=["latitude", "longitude"])

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("regr", Lasso())
])

grid = {
    "regr__alpha": [0.0005, 0.001, 0.005, 0.01, 0.05, 0.1]
}

# cols = ["rank_test_score","mean_test_score","mean_train_score","params"]
gs = GridSearchCV(model, param_grid=grid, cv=5)
gs.fit(X_train, y_train)

pd.DataFrame(gs.cv_results_).sort_values("mean_test_score", ascending=False)

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler(with_mean=True, with_std=True)), # Spiega impatto di with_mean=True e with_std=True
    ("regr", Lasso(alpha=0.0005))
])
model.fit(X_train, y_train);
print_lasso_features_eval(model, preprocessor)

Voglio diminuire feature inutili, performance rimangono accettabili

In [ ]:
model_lasso = Pipeline([
    ("scaler", StandardScaler(with_mean=False)), # Spiega impatto di with_mean=True e with_std=True
    ("regr", Lasso(alpha=0.01))
])
model_lasso.fit(X_train, y_train);
print_lasso_features_eval(model_lasso, preprocessor)
print_eval(X_train, y_train, model_lasso)

In [ ]:
lasso_coefs = model_lasso.named_steps['regr'].coef_

# Indici delle feature con coefficiente diverso da zero
kept_indices = [i for i, coef in enumerate(lasso_coefs) if coef != 0]

X_train_poly_input = X_train[:, kept_indices]
X_val_poly_input = X_val[:, kept_indices]

print(f"Shape finale per il Ridge Polinomiale: {X_train_poly_input.shape}")

In [ ]:
# Modello Ridge
from sklearn.linear_model import Ridge
model = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("regr", Ridge())
])

grid = {
    "regr__alpha": [0.1, 1, 10, 50, 100, 500, 1000]
}

gs = GridSearchCV(model, param_grid=grid, cv=5)
gs.fit(X_train, y_train)

pd.DataFrame(gs.cv_results_).sort_values("mean_test_score", ascending=False)

Proviamo ora a fare una regressione Ridge con feature polinomiali di grado 2, per vedere se è in grado di catturare meglio le relazioni tra le feature e il prezzo, e quindi migliorare la performance del modello di regressione. Solo con le feature estratte da Lasso

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

model_poly = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler(with_mean=False)),
    ("regr", Ridge())
])

grid = {
    "poly__degree": [2],
    "regr__alpha": [100, 200, 300, 500, 1000]
}

gs_poly = GridSearchCV(model_poly, param_grid=grid, cv=5)
gs_poly.fit(X_train_poly_input, y_train)

pd.DataFrame(gs_poly.cv_results_).sort_values("mean_test_score", ascending=False)

In [ ]:
print_eval(X_val_poly_input, y_val, gs_poly.best_estimator_)

In [ ]:
# from sklearn.preprocessing import PolynomialFeatures

# model_poly = Pipeline([
#     ("poly", PolynomialFeatures(include_bias=False)),
#     ("scaler", StandardScaler(with_mean=False)),
#     ("regr", Lasso())
# ])

# grid = {
#     "poly__degree": [2],
#     "regr__alpha": [0.005, 0.01, 0.05]
# }

# gs_poly = GridSearchCV(model_poly, param_grid=grid, cv=5)
# gs_poly.fit(X_train_poly_input, y_train)

# pd.DataFrame(gs_poly.cv_results_).sort_values("mean_test_score", ascending=False)

Definiamo una funzione

In [ ]:
from sklearn.metrics import r2_score

def safe_relative_error(y_real, y_pred):
    y_real = np.asarray(y_real)
    y_pred = np.asarray(y_pred)
    denominator = np.where(np.abs(y_real) < 1e-8, np.nan, np.abs(y_real))
    return np.nanmean(np.abs(y_real - y_pred) / denominator)


def evaluate_regression_model(model, X_train, y_train, X_val, y_val):
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_r2 = r2_score(y_train, train_pred)
    val_r2 = r2_score(y_val, val_pred)
    val_mse = mean_squared_error(y_val, val_pred)
    val_rmse = np.sqrt(val_mse)
    val_relative_error = safe_relative_error(y_val, val_pred)

    return {
        "train_r2": train_r2,
        "val_r2": val_r2,
        "overfit_gap": train_r2 - val_r2,
        "val_mse": val_mse,
        "val_rmse": val_rmse,
        "val_relative_error": val_relative_error,
    }


def print_compact_eval(name, metrics, n_features):
    print(f"{name} con {n_features} feature")
    print(f"       R2 train: {metrics['train_r2']:.4f}")
    print(f"  R2 validation: {metrics['val_r2']:.4f}")
    print(f"   Overfit gap : {metrics['overfit_gap']:.4f}")
    print(f" validation MSE: {metrics['val_mse']:.4f}")
    print(f" validation RMSE: {metrics['val_rmse']:.4f}")
    print(f" validation relative error: {metrics['val_relative_error']:.2%}")

    if metrics["overfit_gap"] > 0.10:
        print("\tOsservazione: gap train-validation alto, possibile overfitting.")
    else:
        print("\tOsservazione: gap contenuto, il modello sembra stabile.")


def print_eval(model, X_train, y_train, X_val, y_val):
    metrics = evaluate_regression_model(model, X_train, y_train, X_val, y_val)
    print_compact_eval("Modello", metrics, X_train.shape[1])


### XGBOOST PARTE NICO

Fino ad ora abbiamo utilizzato modelli di regressione lineare e polinomiali, che però non sono in grado di catturare le relazioni non lineari complesse tra le feature e il prezzo. 

Proviamo ora ad utilizzare dei modelli di regressione non lineare basati su alberi di decisione:
*Spiega modelli di regressione non lineare*

Iniziamo con il modello DecisionTreeRegressor, di scikit-learn, che è un modello di regressione basato su alberi di decisione.

In [ ]:
X_train, X_val, y_train, y_val, cols, preprocessor = preprocess_data_w_target_encoder(lst_for_price_preproc, cols_to_drop=["latitude", "longitude"])

In [ ]:
lst_for_price_preproc.info()

In [ ]:
SEED = 7112004

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model_dt = DecisionTreeRegressor(
    random_state=SEED
)

grid_dt = {
    'max_depth':        [4, 6, 8, 12, None],
    'min_samples_leaf': [5, 10, 20, 50],
    'max_features':     [0.5, 0.7, 1.0],
}

gs_dt = GridSearchCV(model_dt, param_grid=grid_dt, cv=5, n_jobs=-1, verbose=1)
gs_dt.fit(X_train, y_train)

pd.DataFrame(gs_dt.cv_results_).sort_values("mean_test_score", ascending=False).head(5)

In [ ]:
print_eval(gs_dt.best_estimator_, X_train, y_train, X_val, y_val)

#### Random Forest Regressor
Proviamo ora ad utilizzare un modello di regressione basato su Random Forest, che è un ensemble di alberi di decisione.

Siccome è più complesso di un unico albero di decisione, non è efficente fare una grid search sui parametri. Quindi si è deciso di usare una tecnica di cross validation basata sulla selezione random. Usiamo quindi la classe `RandomizedSearchCV` di scikit-learn per effettuare la ricerca sui parametri del modello di regressione Random Forest.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

model_rf = RandomForestRegressor(
    random_state=SEED,
    n_jobs=-1,
)

distributions_rf = {
    'n_estimators':    randint(100, 500),
    'max_depth':       [8, 12, 20, None],
    'min_samples_leaf': randint(5, 30),
    'max_features':    [0.2, 0.33, 0.5, 0.7],
}

rs_rf = RandomizedSearchCV(model_rf, param_distributions=distributions_rf, n_iter=20, cv=5, n_jobs=-1, random_state=SEED, verbose=1)
rs_rf.fit(X_train, y_train)

pd.DataFrame(rs_rf.cv_results_).sort_values("mean_test_score", ascending=False).head(5)

In [ ]:
print_eval(rs_rf.best_estimator_, X_train, y_train, X_val, y_val)

Questi sono i parametri migliori trovati dalla ricerca randomizzata:

In [ ]:
pd.DataFrame(rs_rf.cv_results_).sort_values("mean_test_score", ascending=False).head(1)["params"].values[0]

In [ ]:
model_rf = RandomForestRegressor(
    random_state=SEED,
    n_jobs=-1,
    max_depth=20,
    max_features=0.7,
    min_samples_leaf=5,
    n_estimators=145
)

model_rf.fit(X_train, y_train)

In [ ]:
print_eval(model_rf, X_train, y_train, X_val, y_val)

#### XGBoost

In [ ]:
from xgboost import XGBRegressor
from scipy.stats import uniform

model_xgb = XGBRegressor(
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist', # Usa l'algoritmo 'hist' per velocizzare l'addestramento su dataset di grandi dimensioni
    early_stopping_rounds=50
)

distributions_xgb = {
    'n_estimators':     [500, 800, 1200],
    'learning_rate':    uniform(0.01, 0.07),  # tra 0.01 e 0.08
    'max_depth':        [5, 6, 7, 8],         # 4 probabilmente troppo poco
    'subsample':        uniform(0.7, 0.3),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha':        [0, 0.1, 0.5],
    'reg_lambda':       [1.0, 2.0, 5.0],      # esplora lambda più alto
}

rs_xgb = RandomizedSearchCV(model_xgb, param_distributions=distributions_xgb, n_iter=3, cv=3, n_jobs=-1, random_state=SEED, verbose=100)
rs_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)])

pd.DataFrame(rs_xgb.cv_results_).sort_values("mean_test_score", ascending=False).head(5)

In [ ]:
best_xgb = rs_xgb.best_estimator_
print_eval(best_xgb, X_train, y_train, X_val, y_val)

In [ ]:
rs_xgb.cv_results_["params"][0]

In [ ]:
# Modello con i migliori iperparametri trovati da RandomizedSearchCV, ma overfittato

model_xgb = XGBRegressor(
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=50,
    colsample_bytree=0.8863680559935583,
    learning_rate=0.06826929752448146,
    max_depth=7,
    n_estimators=1200,
    reg_alpha=0,
    reg_lambda=2.0,
    subsample=0.9217050058668048
)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)])


In [ ]:
print_eval(model_xgb, X_train, y_train, X_val, y_val)

In [ ]:
# Modello regolarizzato per cercare di ridurre l'overfitting, con iperparametri scelti a mano partendo da quelli trovati da RandomizedSearchCV

model_xgb = XGBRegressor(
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=50,

    # Abbassa la capacità
    max_depth=5,               # era 7
    colsample_bytree=0.65,     # era 0.886
    subsample=0.80,            # era 0.922

    # Alza la regolarizzazione
    reg_lambda=8.0,            # era 2.0
    reg_alpha=0.5,             # era 0
    min_child_weight=10,       # era default (1)

    # Tieni questi
    learning_rate=0.068,
    n_estimators=1200,
)
model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)])

In [ ]:
print_eval(model_xgb, X_train, y_train, X_val, y_val)

In [ ]:
from scipy.stats import uniform, randint

distributions_xgb_focused = {
    # I tre parametri che controllano il gap — unici da cercare
    'max_depth':        [5, 6, 7],
    'min_child_weight': [3, 5, 7, 10],
    'reg_lambda':       [3.0, 5.0, 8.0],
    'reg_alpha':        [0, 0.2, 0.5],
}

xgb_focused = XGBRegressor(
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=50,

    # Fissi — già ottimizzati
    learning_rate=0.068,
    n_estimators=1200,
    colsample_bytree=0.75,   # punto medio tra 0.65 e 0.886
    subsample=0.85,          # punto medio tra 0.80 e 0.922
)

rscv_focused = RandomizedSearchCV(
    xgb_focused,
    distributions_xgb_focused,
    n_iter=8,          # 8 × 3 folds × 21s ≈ 504s → ~8 minuti
    cv=3,              # invece di 5
    scoring='r2',
    random_state=SEED,
    n_jobs=1,          # XGBoost usa già n_jobs=-1 internamente
    verbose=1
)

rscv_focused.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

pd.DataFrame(rscv_focused.cv_results_).sort_values("mean_test_score", ascending=False)

In [ ]:
rscv_focused.cv_results_["params"][0]

In [ ]:
lst_for_price_preproc_xgb = build_price_preprocessing_dataframe(listings, top_n_property_types=15, top_n_amenities=200)

In [ ]:
X_train, X_val, y_train, y_val, cols, preprocessor = preprocess_data_w_target_encoder(lst_for_price_preproc_xgb, cols_to_drop=["latitude", "longitude"])

In [ ]:
model_xgb_best = XGBRegressor(
    random_state=SEED,
    n_jobs=-1,
    tree_method='hist',
    early_stopping_rounds=50,
    learning_rate=0.068,
    n_estimators=1200,
    colsample_bytree=0.75,
    subsample=0.85,
    reg_lambda=8.0,
    reg_alpha=0.5,
    min_child_weight=3,
    max_depth=5
)

model_xgb_best.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

print_eval(model_xgb_best, X_train, y_train, X_val, y_val)

In [ ]:
# Prendi il best estimator dal RSCV


# Feature importance come DataFrame
importances = pd.DataFrame({
    'importance': best_xgb.feature_importances_
}).sort_values('importance', ascending=False)

print(importances.head(10))

### XGBOOST PARTE MATTE CECCA

#### Modelli compatti con Random Forest e XGBoost

In questa sezione proviamo modelli ad albero piu flessibili della Lasso, ma li costruiamo con una regola precisa: **meno variabili possibile e controllo dell'overfitting**.

L'idea e questa:
1. partiamo dalle feature gia preprocessate in `X_train` e `X_val`;
2. usiamo la Lasso gia addestrata (`model_lasso`) come filtro iniziale, tenendo solo le feature con coefficiente diverso da zero;
3. proviamo sottoinsiemi piccoli da 10, 20, 30 e 50 variabili;
4. addestriamo Random Forest e XGBoost con iperparametri regolarizzati;
5. scegliamo il modello con il miglior compromesso tra `R2` su validation set e numero di feature.

Per una regressione non si parla davvero di accuracy come nella classificazione. Qui useremo quindi il **R2 sul validation set** come indicatore principale: piu e alto, meglio il modello spiega la variabilita del prezzo.


In [ ]:
# Step 0 - Import e installazione di XGBoost se manca.
# XGBoost non e una dipendenza standard di scikit-learn: se il pacchetto non e presente,
# questa cella lo installa nell'ambiente Python usato dal notebook.

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("xgboost") is None:
    print("XGBoost non trovato: installazione in corso...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
else:
    print("XGBoost gia disponibile.")

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor

SEED = 7112004


##### Step 1 - Funzioni di valutazione

Valutiamo sempre il modello sia su training set sia su validation set. Questo confronto e fondamentale:

- se `train_r2` e alto ma `val_r2` e molto piu basso, il modello sta probabilmente overfittando;
- se `train_r2` e `val_r2` sono vicini, il modello generalizza meglio;
- tra due modelli con prestazioni simili, preferiamo quello con meno feature.


In [ ]:
# Step 1 - Metriche compatte per confrontare i modelli.
# Usiamo R2 come metrica principale, ma teniamo anche RMSE ed errore relativo
# per capire quanto il modello sbaglia in media.

def safe_relative_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denominator = np.where(np.abs(y_true) < 1e-8, np.nan, np.abs(y_true))
    return np.nanmean(np.abs(y_true - y_pred) / denominator)


def evaluate_regression_model(model, X_train_data, y_train_data, X_val_data, y_val_data):
    train_pred = model.predict(X_train_data)
    val_pred = model.predict(X_val_data)

    train_r2 = r2_score(y_train_data, train_pred)
    val_r2 = r2_score(y_val_data, val_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val_data, val_pred))
    val_relative_error = safe_relative_error(y_val_data, val_pred)

    return {
        "train_r2": train_r2,
        "val_r2": val_r2,
        "overfit_gap": train_r2 - val_r2,
        "val_rmse": val_rmse,
        "val_relative_error": val_relative_error,
    }


def print_compact_eval(name, metrics, n_features):
    print(f"{name} con {n_features} feature")
    print(f"       R2 train: {metrics['train_r2']:.4f}")
    print(f"  R2 validation: {metrics['val_r2']:.4f}")
    print(f"   Overfit gap : {metrics['overfit_gap']:.4f}")
    print(f" validation RMSE: {metrics['val_rmse']:.4f}")
    print(f" validation relative error: {metrics['val_relative_error']:.2%}")

    if metrics["overfit_gap"] > 0.10:
        print("  Osservazione: gap train-validation alto, possibile overfitting.")
    else:
        print("  Osservazione: gap contenuto, il modello sembra piu stabile.")


##### Step 2 - Selezione di poche feature con Lasso

La Lasso e utile come filtro perche tende a portare a zero i coefficienti delle feature meno utili. Qui usiamo quindi `model_lasso` non come modello finale, ma come selettore iniziale.

Poi non prendiamo tutte le feature rimaste: proviamo solo i migliori sottoinsiemi piccoli, ordinati per valore assoluto del coefficiente Lasso.


In [ ]:
# Step 2 - Recupero dei nomi delle feature preprocessate.
# Il controllo sulla lunghezza evita errori se il preprocessor non restituisce i nomi
# nello stesso numero delle colonne trasformate.

feature_names = np.array(preprocessor.get_feature_names_out())

if len(feature_names) != X_train.shape[1]:
    print("Attenzione: numero nomi feature diverso dal numero colonne. Uso nomi generici.")
    feature_names = np.array([f"feature_{i}" for i in range(X_train.shape[1])])

lasso_coefs = model_lasso.named_steps["regr"].coef_

if len(lasso_coefs) != X_train.shape[1]:
    raise ValueError("I coefficienti della Lasso non hanno la stessa dimensione di X_train.")

# Prendiamo prima le feature non azzerate dalla Lasso.
non_zero_indices = np.flatnonzero(lasso_coefs != 0)

# Se per qualche motivo la Lasso fosse troppo aggressiva, usiamo tutte le feature
# ordinate per importanza assoluta, cosi il resto del tutorial continua a funzionare.
if len(non_zero_indices) == 0:
    print("La Lasso ha azzerato tutte le feature: uso tutte le feature ordinate per peso assoluto.")
    non_zero_indices = np.arange(len(lasso_coefs))

# Ordiniamo le feature candidate: prima quelle con coefficiente Lasso piu grande in valore assoluto.
ordered_lasso_indices = non_zero_indices[np.argsort(np.abs(lasso_coefs[non_zero_indices]))[::-1]]

# Proviamo modelli sempre piu piccoli/grandi, ma senza superare il numero di feature disponibili.
feature_counts_to_try = sorted({min(n, len(ordered_lasso_indices)) for n in [10, 20, 30, 50]})

print(f"Feature totali preprocessate: {X_train.shape[1]}")
print(f"Feature tenute dalla Lasso: {len(non_zero_indices)}")
print(f"Dimensioni di feature che proveremo: {feature_counts_to_try}")

pd.Series(
    np.abs(lasso_coefs[ordered_lasso_indices]),
    index=feature_names[ordered_lasso_indices]
).head(20)


##### Step 3 - Ricerca di modelli piccoli

Per ogni numero di feature proviamo due famiglie di modelli:

- **Random Forest compatta**: pochi alberi, profondita limitata e foglie piu grandi. Questo riduce la capacita del modello di imparare rumore.
- **XGBoost compatto**: alberi poco profondi e regolarizzazione (`reg_lambda`, `reg_alpha`). Questo aiuta a ottenere buon segnale senza far crescere troppo il modello.

Gli alberi non richiedono standardizzazione delle variabili, quindi usiamo direttamente le feature preprocessate.


In [ ]:
# Step 3 - Griglie compatte per modelli piccoli.
# Usiamo KFold con shuffle per rendere la validazione interna piu stabile.

cv = KFold(n_splits=3, shuffle=True, random_state=SEED)

rf_model = RandomForestRegressor(
    n_jobs=-1,
    random_state=SEED,
)

rf_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 8, 12],
    "min_samples_leaf": [3, 5, 10],
    "max_features": ["sqrt", 0.5],
}

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)

xgb_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.6, 0.8],
    "reg_lambda": [5, 10],
    "reg_alpha": [0, 0.1],
}


In [ ]:
# Step 4 - Addestramento e confronto.
# Per ogni dimensione del subset creiamo X_train_small e X_val_small.
# Salviamo anche estimator e nomi delle feature, cosi alla fine possiamo interpretare il modello migliore.

result_rows = []
trained_models = {}
selected_features_by_model = {}
selected_indices_by_model = {}

for n_features in feature_counts_to_try:
    selected_indices = ordered_lasso_indices[:n_features]
    selected_feature_names = feature_names[selected_indices]

    X_train_small = X_train[:, selected_indices]
    X_val_small = X_val[:, selected_indices]

    if X_train_small.shape[1] != len(selected_feature_names):
        raise ValueError("Il numero di colonne selezionate non coincide con i nomi delle feature.")

    print("=" * 80)
    print(f"Provo modelli con {n_features} feature")
    print("Prime feature usate:", list(selected_feature_names[:10]))

    # Random Forest compatta.
    rf_search = GridSearchCV(
        estimator=rf_model,
        param_grid=rf_grid,
        scoring="r2",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    rf_search.fit(X_train_small, y_train)
    best_rf = rf_search.best_estimator_
    rf_metrics = evaluate_regression_model(best_rf, X_train_small, y_train, X_val_small, y_val)
    print_compact_eval("Random Forest", rf_metrics, n_features)

    rf_key = f"RandomForest_{n_features}_features"
    trained_models[rf_key] = best_rf
    selected_features_by_model[rf_key] = selected_feature_names
    selected_indices_by_model[rf_key] = selected_indices
    result_rows.append({
        "model_key": rf_key,
        "model": "Random Forest",
        "n_features": n_features,
        "best_params": rf_search.best_params_,
        **rf_metrics,
    })

    # XGBoost compatto.
    xgb_search = GridSearchCV(
        estimator=xgb_model,
        param_grid=xgb_grid,
        scoring="r2",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    xgb_search.fit(X_train_small, y_train)
    best_xgb = xgb_search.best_estimator_
    xgb_metrics = evaluate_regression_model(best_xgb, X_train_small, y_train, X_val_small, y_val)
    print_compact_eval("XGBoost", xgb_metrics, n_features)

    xgb_key = f"XGBoost_{n_features}_features"
    trained_models[xgb_key] = best_xgb
    selected_features_by_model[xgb_key] = selected_feature_names
    selected_indices_by_model[xgb_key] = selected_indices
    result_rows.append({
        "model_key": xgb_key,
        "model": "XGBoost",
        "n_features": n_features,
        "best_params": xgb_search.best_params_,
        **xgb_metrics,
    })

model_results = pd.DataFrame(result_rows)
model_results.sort_values(["val_r2", "n_features"], ascending=[False, True])


##### Step 4 - Scelta del miglior compromesso

Non scegliamo automaticamente il modello piu grande. Prima cerchiamo il miglior `R2` sul validation set, poi consideriamo competitivi anche i modelli entro 0.01 punti di R2 dal migliore.

Dentro questa fascia, scegliamo il modello con meno feature. Questa e una scelta conservativa: un modello leggermente piu piccolo ma quasi altrettanto accurato di solito e piu robusto.


In [ ]:
# Step 5 - Scelta automatica del miglior compromesso tra accuratezza e semplicita.

r2_tolerance = 0.01
best_val_r2 = model_results["val_r2"].max()

competitive_models = model_results[
    model_results["val_r2"] >= best_val_r2 - r2_tolerance
].copy()

best_row = competitive_models.sort_values(
    ["n_features", "overfit_gap", "val_rmse"],
    ascending=[True, True, True]
).iloc[0]

best_model_key = best_row["model_key"]
best_model = trained_models[best_model_key]
best_feature_names = selected_features_by_model[best_model_key]
best_selected_indices = selected_indices_by_model[best_model_key]

# Rendiamo disponibili le matrici ridotte del modello migliore anche fuori dal ciclo.
X_train_small = X_train[:, best_selected_indices]
X_val_small = X_val[:, best_selected_indices]
selected_feature_names = best_feature_names

print("Miglior compromesso trovato")
print(f"Modello: {best_row['model']}")
print(f"Feature usate: {best_row['n_features']}")
print(f"R2 validation: {best_row['val_r2']:.4f}")
print(f"RMSE validation: {best_row['val_rmse']:.4f}")
print(f"Errore relativo validation: {best_row['val_relative_error']:.2%}")
print(f"Overfit gap: {best_row['overfit_gap']:.4f}")
print("Parametri migliori:")
print(best_row["best_params"])

model_results.sort_values(["val_r2", "n_features"], ascending=[False, True])[
    ["model", "n_features", "train_r2", "val_r2", "overfit_gap", "val_rmse", "val_relative_error", "best_params"]
]


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

# Predizioni sul validation set
y_val_pred = best_model.predict(X_val_small)

# Metriche principali
val_r2 = r2_score(y_val, y_val_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

# Errore relativo medio, evitando divisioni per zero
eps = 1e-9
val_relative_error = np.mean(np.abs(y_val - y_val_pred) / np.maximum(np.abs(y_val), eps))

# RMSPE, utile se il target è sempre positivo
val_rmspe = np.sqrt(
    np.mean(
        ((y_val - y_val_pred) / np.maximum(np.abs(y_val), eps)) ** 2
    )
)

print("Performance modello migliore su validation")
print(f"Modello: {best_row['model']}")
print(f"Feature usate: {best_row['n_features']}")
print(f"R²: {val_r2:.4f}")
print(f"MAE: {val_mae:.4f}")
print(f"RMSE: {val_rmse:.4f}")
print(f"Errore relativo medio: {val_relative_error:.2%}")
print(f"RMSPE: {val_rmspe:.2%}")

##### Step 5 - Interpretazione delle feature importanti

Gli alberi forniscono `feature_importances_`: valori piu alti indicano feature usate piu spesso o in modo piu utile per ridurre l'errore.

Questa non e una prova causale, ma e molto utile per capire quali variabili stanno guidando il modello finale.


In [ ]:
# Step 6 - Feature importance del modello scelto.

feature_importance = pd.Series(
    best_model.feature_importances_,
    index=best_feature_names,
).sort_values(ascending=False)

print("Top feature del modello migliore:")
display(feature_importance.head(20))

plt.figure(figsize=(10, 6))
feature_importance.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 feature importance - modello compatto migliore")
plt.xlabel("Importanza")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


##### Osservazioni finali e nuove variabili da considerare

Per migliorare ancora il modello senza renderlo enorme, conviene creare poche feature nuove ma molto informative:

- **Host**: `host_is_superhost`, `host_response_rate`, `host_acceptance_rate`, `host_identity_verified`, `host_listings_count`.
- **Disponibilita**: `availability_30`, `availability_90`, `availability_365`, `minimum_nights`, `maximum_nights`.
- **Recensioni**: `number_of_reviews`, `reviews_per_month`, `review_scores_rating`, `review_scores_location`, `review_scores_value`.
- **Tempo**: anni da `host_since`, giorni dall'ultima recensione, mese o stagione dello scraping.
- **Geografia**: distanza da centro, stazione, universita e punti turistici; densita di punti di interesse entro raggi diversi.
- **Testo semplice**: lunghezza della descrizione, numero di parole nel titolo, presenza di parole chiave come `parking`, `balcony`, `center`, `wifi`.

La cosa importante e aggiungerle poche alla volta: dopo ogni gruppo di nuove variabili si riesegue la selezione Lasso e si controlla se il `val_r2` migliora senza aumentare troppo l'overfit gap.
